# Capítulo 9: Classificação

**Bases 5 — Ciência de Dados** · notebook de aula

Cada célula de código é a mesma do livro e roda na ordem em que aparece — execute de cima para baixo. Versão publicada deste capítulo: [https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/index.html](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/index.html)

> **Gerado automaticamente a partir dos `.qmd` do livro por `scripts/gerar-notebooks.py`.** Edições feitas aqui se perdem no próximo `make notebooks`; para mudar o conteúdo, edite o `.qmd`.

In [ ]:
# Põe o diretório de trabalho na raiz do projeto — é o que faz
# `from scratch...` e os caminhos `dados/...` funcionarem. No livro isso vem
# do `execute-dir: project` do Quarto; aqui é feito à mão.
#
# No Colab não existe cópia do projeto, então esta célula clona uma. É rápido
# (clone raso) e acontece só na primeira execução da sessão.
import os
import subprocess
import sys

REPO = "https://github.com/BragaD/UnDF-Bases5-CienciaDeDados-202602.git"


def raiz_do_projeto(inicio="."):
    """Sobe os diretórios até achar o `_quarto.yml`. None se não houver."""
    atual = os.path.abspath(inicio)
    while not os.path.exists(os.path.join(atual, "_quarto.yml")):
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai
    return atual


raiz = raiz_do_projeto()
if raiz is None:
    destino = "/content/bases5" if os.path.isdir("/content") else "bases5"
    if not os.path.isdir(destino):
        print("baixando o material da disciplina...")
        subprocess.run(["git", "clone", "--depth", "1", REPO, destino], check=True)
    raiz = raiz_do_projeto(destino)

os.chdir(raiz)
if raiz not in sys.path:
    sys.path.insert(0, raiz)

%matplotlib inline
print("diretório de trabalho:", os.getcwd())

> **📌 Nota**
>
> Este capítulo corresponde ao capítulo 4 de James et al. (2023).

> Vê-se, por este Ensaio, que a teoria das probabilidades não é, no fundo, senão o bom senso reduzido ao cálculo; ela faz apreciar com exatidão o que os espíritos justos sentem por uma espécie de instinto, sem que, muitas vezes, consigam dar-se conta disso.
>
> — Pierre-Simon Laplace

Em `Default`, com dez mil clientes de cartão de crédito, ficar inadimplente ou não é uma categoria, e não uma quantidade. Como prever uma categoria? A seção 9.1 mostra por que uma reta ajustada a esse alvo falha: ela prevê valores fora do intervalo em que uma probabilidade pode existir. O problema muda de forma, sem desaparecer, quando a resposta tem mais de duas categorias sem ordem natural entre si. A seção 9.2 apresenta o remédio, a regressão logística: uma curva que nunca sai de [0, 1], com coeficientes que se leem em log-chance, e não como efeito direto sobre a probabilidade. Com ela, a seção desfaz o paradoxo do estudante, que parece mais arriscado sozinho e menos arriscado quando o saldo entra na conta.

A seção 9.3 estende a mesma ideia para respostas com mais de duas classes, como a origem de um carro em `Auto`. O `scikit-learn` ajusta a parametrização *softmax*, simétrica entre as classes, e ela diz o mesmo que a parametrização com uma classe-base. A seção 9.4 muda de caminho: em vez de modelar direto a probabilidade de $Y$, LDA, QDA e Naive Bayes modelam como $X$ se distribui dentro de cada classe e usam o teorema de Bayes para inverter essa distribuição em probabilidade. Cada método faz uma suposição diferente sobre essa distribuição, e o preço de cada suposição, certa ou errada, é medido em simulação, com a fronteira verdadeira conhecida.

As duas últimas seções trocam de pergunta: não mais como ajustar um classificador, mas como julgá-lo. A seção 9.5 mostra que a acurácia sozinha engana quando a classe de interesse é rara, e monta o vocabulário que a substitui: matriz de confusão, precisão, revocação, a escolha do limiar e a curva ROC, que resume de uma vez todos os limiares possíveis. A seção 9.6 compara os classificadores do capítulo (regressão logística, LDA, QDA, Naive Bayes e *k*-NN com dois tamanhos de vizinhança) em cenários simulados com fronteira conhecida. A resposta da comparação é "depende": de qual suposição a fronteira verdadeira confirma ou viola, e de quanto dado há para pagar o preço de uma fronteira mais flexível.

Ao final deste capítulo, você será capaz de:

- Reconhecer por que uma reta ajustada sobre `Default` prevê fora do intervalo [0, 1] quando o alvo é qualitativo, e por que codificar como números uma resposta de mais de duas categorias impõe uma ordem e uma distância que ela não tem
- Ajustar uma regressão logística sobre `Default`, ler o coeficiente como efeito sobre a log-chance (e não sobre a probabilidade) e explicar o paradoxo do estudante pela associação entre `estudante` e `saldo`
- Estender a regressão logística a mais de duas classes, como a `origem` de um carro em `Auto`, pela parametrização *softmax* do `scikit-learn`, e relacioná-la à parametrização com classe-base
- Distinguir LDA, QDA e Naive Bayes pela suposição que cada um faz sobre a distribuição de $X$ dentro de cada classe, e relacionar o custo de uma suposição, certa ou errada, ao compromisso entre viés e variância
- Avaliar um classificador além da acurácia, pela matriz de confusão, precisão, revocação, escolha do limiar e curva ROC, reconhecendo que a acurácia sozinha engana quando a classe de interesse é rara
- Comparar regressão logística, LDA, QDA, Naive Bayes e *k*-NN em cenários simulados de fronteira conhecida e sobre `Default`, reconhecendo que a vantagem de cada método depende de qual suposição a fronteira verdadeira confirma ou viola

## Seções

| Seção | Tópico |
|---|---|
| [9.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/01-por-que-nao-regressao-linear.html) | Por que Não Regressão Linear |
| [9.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/02-regressao-logistica.html) | Regressão Logística |
| [9.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/03-logistica-multinomial.html) | Logística Multinomial |
| [9.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/04-modelos-generativos.html) | Modelos Generativos: LDA, QDA e Naive Bayes |
| [9.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/05-avaliando-um-classificador.html) | Avaliando um Classificador |
| [9.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/06-comparando-os-metodos.html) | Comparando os Métodos |

## Por que Não Regressão Linear

> **📌 Nota**
>
> Esta seção corresponde às seções 4.1 e 4.2 de James et al. (2023).

Dez mil clientes de cartão de crédito, e a pergunta é se cada um fica inadimplente ou não. Em `Default`, `inadimplente` não é uma quantidade: é uma categoria, `sim` ou `não`. `saldo`, `renda` e `estudante` (`sim`/`não`) são preditores como os de sempre; o que muda é o alvo, e é essa mudança que o resto do capítulo resolve.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, LogisticRegression

plt.style.use("estilo-figuras.mplstyle")

### O `Default`: dez mil clientes, um alvo qualitativo

In [ ]:
base = pd.read_csv("dados/Default.csv")
base.shape, base.columns.tolist()

Dez mil linhas e quatro colunas. `inadimplente` é o alvo, `estudante` é uma segunda coluna categórica, e `saldo` (quanto o cliente deve no cartão) e `renda` (a renda anual do cliente) são numéricas, as duas em dólares.

In [ ]:
n_sim = int((base["inadimplente"] == "sim").sum())
n_nao = int((base["inadimplente"] == "não").sum())
proporcao_sim = n_sim / len(base)

n_sim, n_nao, round(proporcao_sim, 4)

333 dos 10.000 clientes ficam inadimplentes, e 9.667 não. A proporção, 0,0333, é baixa: cerca de um em trinta. `Default` é um conjunto desequilibrado, e um classificador que sempre responda `não` já acerta a maioria. O ponto volta na seção 9.5, onde "acertar a maioria" deixa de bastar como medida.

### O saldo separa; a renda, não

In [ ]:
# Figura: `Default`: saldo e renda de 10.000 clientes, com quem ficou inadimplente (`sim`, laranja) destacado sobre quem não ficou (`não`, azul). Ao lado, os mesmos dois grupos em caixas (quartil de 25%, mediana e quartil de 75%) para saldo e para renda.
nao = base[base["inadimplente"] == "não"]
sim = base[base["inadimplente"] == "sim"]


def desenha_caixas(ax, dados_nao, dados_sim, rotulo_y):
    for dados, posicao, cor in [(dados_nao, 1, "C0"), (dados_sim, 2, "C1")]:
        propriedades = dict(color=cor, linewidth=1.6)
        ax.boxplot(
            dados,
            positions=[posicao],
            widths=0.5,
            boxprops=propriedades,
            whiskerprops=propriedades,
            capprops=propriedades,
            medianprops=propriedades,
            flierprops=dict(markeredgecolor=cor, markersize=3),
        )
    ax.set_xticks([1, 2])
    ax.set_xticklabels(["não", "sim"])
    ax.set_xlabel("inadimplente")
    ax.set_ylabel(rotulo_y)
    ax.set_ylim(bottom=0)


fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(12, 4.4))

ax1.scatter(nao["saldo"], nao["renda"], color="C0", s=3, linewidths=0, label="não")
ax1.scatter(
    sim["saldo"], sim["renda"], color="C1", s=9, linewidths=0, label="sim", zorder=3
)
ax1.set_xlim(left=0)
ax1.set_ylim(bottom=0)
ax1.set_xlabel("saldo (dólares)")
ax1.set_ylabel("renda (dólares)")
ax1.legend(
    title="inadimplente",
    loc="upper left",
    bbox_to_anchor=(1.0, 1.0),
    markerscale=1.6,
)

desenha_caixas(ax2, nao["saldo"], sim["saldo"], "saldo (dólares)")
desenha_caixas(ax3, nao["renda"], sim["renda"], "renda (dólares)")

plt.tight_layout()
plt.show()

> **🔧 Função**
>
> **`ax.boxplot(dados, positions, widths)`**: desenha uma caixa com os quartis de 25% e 75% nas bordas e a mediana no meio, com hastes até os valores mais afastados que ainda não contam como atípicos. `positions` põe a caixa numa posição do eixo horizontal, e os `...props` escolhem a cor de cada parte.

A dispersão à esquerda já sugere uma leitura: os pontos laranja (`sim`) se acumulam à direita, em saldos altos, e os azuis (`não`), à esquerda. Em renda, as duas cores se misturam ao longo de todo o eixo vertical. As caixas ao lado dão para conferir essa leitura com números:

In [ ]:
q1_saldo_nao = float(nao["saldo"].quantile(0.25))
q3_saldo_nao = float(nao["saldo"].quantile(0.75))
q1_saldo_sim = float(sim["saldo"].quantile(0.25))
q3_saldo_sim = float(sim["saldo"].quantile(0.75))
sobrepoe_saldo = not (q3_saldo_nao < q1_saldo_sim or q3_saldo_sim < q1_saldo_nao)

q1_renda_nao = float(nao["renda"].quantile(0.25))
q3_renda_nao = float(nao["renda"].quantile(0.75))
q1_renda_sim = float(sim["renda"].quantile(0.25))
q3_renda_sim = float(sim["renda"].quantile(0.75))
sobrepoe_renda = not (q3_renda_nao < q1_renda_sim or q3_renda_sim < q1_renda_nao)

{
    "saldo (não: q1, q3)": (round(q1_saldo_nao, 2), round(q3_saldo_nao, 2)),
    "saldo (sim: q1, q3)": (round(q1_saldo_sim, 2), round(q3_saldo_sim, 2)),
    "sobrepoe_saldo": sobrepoe_saldo,
    "renda (não: q1, q3)": (round(q1_renda_nao, 2), round(q3_renda_nao, 2)),
    "renda (sim: q1, q3)": (round(q1_renda_sim, 2), round(q3_renda_sim, 2)),
    "sobrepoe_renda": sobrepoe_renda,
}

> **🔧 Função**
>
> **`serie.quantile(q)`**: o valor abaixo do qual fica a fração `q` dos dados. `quantile(0.25)` e `quantile(0.75)` são os quartis que formam as bordas da caixa.

Em `saldo`, o quartil de 75% de quem não ficou inadimplente (1.128,25) fica abaixo do quartil de 25% de quem ficou (1.511,61). As duas caixas nem se tocam, e `sobrepoe_saldo` sai `False`. Em `renda`, o miolo de quem não ficou inadimplente vai de 21.405,06 a 43.823,76, e o de quem ficou, de 19.027,51 a 43.067,33, uma faixa quase inteiramente contida na do outro grupo; `sobrepoe_renda` sai `True`. O saldo separa os dois grupos; a renda, não.

### Por que a reta não serve para probabilidade

Recodificando `inadimplente` como 0 (`não`) e 1 (`sim`), nada impede de ajustar a mesma `LinearRegression` do capítulo anterior sobre esse alvo. O método não sabe, e não pergunta, se `y` é uma venda em milhares de unidades ou uma categoria disfarçada de número. O que ele devolve, porém, passa a ser lido como uma probabilidade prevista: a de que aquele cliente fique inadimplente. Só que uma reta de inclinação positiva não para em zero para os saldos mais baixos. O que ela prevê ali?

In [ ]:
y = (base["inadimplente"] == "sim").astype(int)
X = base[["saldo"]]

reta = LinearRegression().fit(X, y)
previsoes = reta.predict(X)

n_negativas = int((previsoes < 0).sum())
minimo_previsto = float(previsoes.min())

saldo_minimo = float(base["saldo"].min())

n_negativas, round(minimo_previsto, 4), round(float(reta.coef_[0]), 6), saldo_minimo

3.123 dos 10.000 clientes recebem da reta uma previsão negativa, quase um em cada três. Como a inclinação é positiva (0,00013 por dólar), a menor previsão, -0,0752, é a de quem tem o menor saldo, que é 0. Probabilidade negativa não tem leitura possível: nenhum cliente tem chance "menos que zero por cento" de ficar inadimplente. Uma fração considerável das previsões da reta cai fora do intervalo em que uma probabilidade pode existir, e não por acaso: é a forma da reta que leva a isso.

### A curva que não sai de [0, 1]

In [ ]:
# Figura: Saldo contra a previsão de inadimplência, para os mesmos clientes de `Default`. Esquerda: a reta ajustada acima, que cruza a faixa sombreada (o intervalo [0, 1]) e sai por baixo dela. Direita: uma curva logística ajustada aos mesmos dados. Nos dois painéis, os traços no alto e embaixo marcam, para cada cliente, se ele ficou inadimplente (no topo) ou não (embaixo).
logistica = LogisticRegression(C=np.inf, solver="newton-cholesky", tol=1e-8).fit(X, y)

grade_saldo = pd.DataFrame(
    {"saldo": np.linspace(base["saldo"].min(), base["saldo"].max(), 300)}
)
reta_grade = reta.predict(grade_saldo)
prob_logistica_grade = logistica.predict_proba(grade_saldo)[:, 1]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.4), sharey=True)

for ax in (ax1, ax2):
    ax.axhspan(0, 1, color="C2", alpha=0.10)
    ax.axhline(0, color="C2", linewidth=1.2)
    ax.axhline(1, color="C2", linewidth=1.2)
    ax.plot(
        nao["saldo"],
        np.zeros(len(nao)),
        "|",
        color="C0",
        markersize=6,
        markeredgewidth=0.6,
    )
    ax.plot(
        sim["saldo"],
        np.ones(len(sim)),
        "|",
        color="C1",
        markersize=6,
        markeredgewidth=0.6,
    )
    ax.set_xlim(left=0)
    ax.set_xlabel("saldo (dólares)")

ax1.plot(grade_saldo["saldo"], reta_grade, color="C3", linewidth=2.2)
ax1.scatter(
    [saldo_minimo], [minimo_previsto], color="C3", s=40, zorder=3, clip_on=False
)
ax1.annotate(
    f"mínimo: {minimo_previsto:.3f}".replace(".", ","),
    xy=(saldo_minimo, minimo_previsto),
    xytext=(14, -6),
    textcoords="offset points",
    fontsize=9,
)
ax1.set_ylabel("previsão / probabilidade de inadimplência")
ax1.set_title("reta")

ax2.plot(grade_saldo["saldo"], prob_logistica_grade, color="C3", linewidth=2.2)
ax2.set_title("curva logística")

plt.tight_layout()
plt.show()

> **🔧 Função**
>
> **`LogisticRegression(C, solver, tol).fit(X, y)`**: ajusta uma regressão logística, a curva em S da seção 9.2.
>
> - `C=np.inf`: desliga a penalização que o `scikit-learn` aplica por padrão, para que o ajuste seja o de máxima verossimilhança.
> - `solver="newton-cholesky"` e `tol=1e-8`: o método de otimização e a tolerância para parar. Com eles, o ajuste chega ao máximo em poucas iterações, com o mesmo resultado em qualquer máquina.
>
> **`np.zeros(n)`** e **`np.ones(n)`**: `n` zeros ou `n` uns, aqui a altura dos traços embaixo e no alto. **`np.linspace(inicio, fim, n)`**: `n` números igualmente espaçados de `inicio` a `fim`, aqui os saldos em que a reta e a curva são desenhadas. **`ax.axhspan(y0, y1)`** sombreia a faixa horizontal entre `y0` e `y1`, aqui o intervalo [0, 1].
>
> **`modelo.predict_proba(X)`**: uma coluna de probabilidades por classe, na ordem de `modelo.classes_`. Aqui as classes são 0 e 1, e `[:, 1]` pega a probabilidade de inadimplência.

A reta (esquerda) atravessa a faixa sombreada e sai por baixo dela para saldos próximos de zero: é a mesma previsão negativa medida acima, agora como curva. A curva logística (direita), ajustada aos mesmos clientes, tem outro formato: um S que se aproxima de 0 e de 1 sem nunca chegar a eles.

In [ ]:
minimo_logistica = float(prob_logistica_grade.min())
maximo_logistica = float(prob_logistica_grade.max())
n_fora_logistica = int(((prob_logistica_grade < 0) | (prob_logistica_grade > 1)).sum())

round(minimo_logistica, 5), round(maximo_logistica, 4), n_fora_logistica

Sobre a mesma grade de 300 valores de saldo usada na figura, a curva logística vai de 0,00002 a 0,9810, perto das bordas sem cruzá-las, e `n_fora_logistica` conta zero pontos fora de [0, 1]. Como a curva se ajusta, e por que ela tem esse formato, é o assunto da seção 9.2.

### Mais de duas classes, e a ordem que a codificação inventa

O problema muda de figura, mas não desaparece, quando o alvo tem mais de duas categorias. Um paciente chega ao pronto-socorro com sintomas que apontam para um de três diagnósticos: derrame, overdose ou convulsão. Que número dar a cada um, para que uma reta possa prevê-los? Codificar isso como $Y = 1$ para derrame, $2$ para overdose e $3$ para convulsão impõe duas coisas que a lista de diagnósticos não tinha: uma ordem entre os três, e a afirmação de que a distância entre derrame e overdose é a mesma que entre overdose e convulsão.

Trocar a ordem ($1$ para convulsão, $2$ para derrame, $3$ para overdose) é uma codificação igualmente válida, e produz um modelo linear diferente do primeiro. Nenhuma das duas é mais correta, porque não existe uma escala numérica por trás do diagnóstico que a codificação possa recuperar. Uma resposta com mais de duas categorias sem ordem natural pede outro tratamento, que é o da seção 9.3.

## Regressão Logística

> **📌 Nota**
>
> Esta seção corresponde às seções 4.3.1, 4.3.2, 4.3.3 e 4.3.4 de James et al. (2023).

A curva em S prevê para `Default` probabilidades que nunca saem de [0, 1]. De onde vem essa curva, e como se lê um coeficiente nela? A regressão logística responde às duas perguntas: ajusta a curva a dado real e dá um coeficiente por preditor, lido de um jeito diferente do coeficiente de uma reta.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

plt.style.use("estilo-figuras.mplstyle")

base = pd.read_csv("dados/Default.csv")
y = (base["inadimplente"] == "sim").astype(int)

### A função logística

Para um preditor $X$ e coeficientes $\beta_0$ e $\beta_1$, a regressão logística modela a probabilidade como

$$
p(X) = \frac{e^{\beta_0 + \beta_1 X}}{1 + e^{\beta_0 + \beta_1 X}}.
$$

O denominador é sempre o numerador mais 1, e os dois são positivos para quaisquer $\beta_0$, $\beta_1$ e $X$ reais. A fração fica presa em (0, 1) pela forma da conta, e não pelo ajuste. É essa propriedade, e não o valor de nenhum coeficiente, que resolve o problema da reta.

In [ ]:
# Figura: A função logística, para $\\beta_0 + \\beta_1 X$ entre -10 e 10. A curva se aproxima de 0 e de 1 sem nunca chegar a eles.
grade_z = np.linspace(-10, 10, 400)
p_z = np.exp(grade_z) / (1 + np.exp(grade_z))

fig, ax = plt.subplots()
ax.axhspan(0, 1, color="C2", alpha=0.10)
ax.plot(grade_z, p_z, color="C3", linewidth=2.2)
ax.set_xlim(-10, 10)
ax.set_ylim(-0.05, 1.05)
ax.set_xlabel(r"$\beta_0 + \beta_1 X$")
ax.set_ylabel(r"$p(X)$")
plt.tight_layout()
plt.show()

In [ ]:
p_extremo_baixo = float(p_z.min())
p_extremo_alto = float(p_z.max())

f"{p_extremo_baixo:.6f}", f"{p_extremo_alto:.6f}"

Em $\beta_0 + \beta_1 X = -10$, a curva vale 0,000045; em $+10$, vale 0,999955. Fica perto das bordas de [0, 1], mas não as toca, por mais que $\beta_0 + \beta_1 X$ se afaste de zero.

### O ajuste sobre saldo

In [ ]:
modelo_saldo = LogisticRegression(C=np.inf, solver="newton-cholesky", tol=1e-8).fit(
    base[["saldo"]], y
)

intercepto_saldo = float(modelo_saldo.intercept_[0])
coef_saldo = float(modelo_saldo.coef_[0, 0])

round(intercepto_saldo, 4), round(coef_saldo, 4)

> **🔧 Função**
>
> **`serie.to_frame(nome)`**, usado mais adiante nesta seção, transforma uma coluna solta numa tabela de uma coluna, porque o `fit` espera `X` como tabela.
>
> **`modelo.intercept_`** e **`modelo.coef_`**: numa `LogisticRegression`, os dois vêm como arrays mesmo com uma resposta só, porque o mesmo estimador também serve para mais de duas classes (seção 9.3). `intercept_` tem uma posição e `coef_` é uma tabela com uma linha e uma coluna por preditor; daí o `[0]` e o `[0, 0]`, que na `LinearRegression` não eram necessários.

Sobre `saldo` sozinho, o ajuste dá intercepto -10,6513 e coeficiente 0,0055. Os coeficientes são os que maximizam a verossimilhança, isto é, os que tornam mais provável o padrão de inadimplências que de fato se observou.

### A leitura do coeficiente: log-chance, não probabilidade

A razão $p(X)/(1 - p(X))$ é a **chance** de $Y = 1$: uma probabilidade de 0,2 é uma chance de 0,2/0,8 = 0,25, "um para quatro". Rearranjando a fórmula da logística, o logaritmo da chance vira uma reta em $X$:

$$
\log\left(\frac{p(X)}{1-p(X)}\right) = \beta_0 + \beta_1 X.
$$

> **🔷 Conceito**
>
> Um aumento de uma unidade em $X$ soma $\beta_1$ à **log-chance** de $Y = 1$, e não à probabilidade $p(X)$. Como a relação entre $p(X)$ e $X$ não é uma reta, o efeito de $X$ sobre a probabilidade depende de onde $X$ já está. Na regressão linear, $\beta_1$ valia o mesmo incremento em qualquer ponto.

Sobre `saldo`, cada dólar a mais soma sempre os mesmos 0,0055 à log-chance de inadimplência. E o efeito sobre a probabilidade? Se a previsão para um saldo de mil dólares for pequena, dobrar o saldo dobra a probabilidade?

In [ ]:
grade_previsao = pd.DataFrame({"saldo": [1000, 2000]})
prob_1000, prob_2000 = modelo_saldo.predict_proba(grade_previsao)[:, 1]

round(float(prob_1000) * 100, 2), round(float(prob_2000) * 100, 2)

Para um saldo de mil dólares, a probabilidade prevista de inadimplência é 0,58%. Para um saldo de dois mil dólares, ela não dobra: sobe para 58,58%. É essa não linearidade que a curva em S carrega, e que uma reta não tem como representar.

### O paradoxo do estudante

In [ ]:
estudantes = base[base["estudante"] == "sim"]
nao_estudantes = base[base["estudante"] == "não"]

taxa_estudante = float((estudantes["inadimplente"] == "sim").mean())
taxa_nao_estudante = float((nao_estudantes["inadimplente"] == "sim").mean())

round(taxa_estudante * 100, 2), round(taxa_nao_estudante * 100, 2)

Entre estudantes, 4,31% ficam inadimplentes; entre os demais, 2,92%. Olhado sozinho, o estudante é o cliente mais arriscado dos dois.

In [ ]:
X_estudante = (base["estudante"] == "sim").astype(int).to_frame("estudante_sim")
modelo_estudante = LogisticRegression(
    C=np.inf, solver="newton-cholesky", tol=1e-8
).fit(X_estudante, y)

coef_estudante_sozinho = float(modelo_estudante.coef_[0, 0])
round(coef_estudante_sozinho, 4)

Um modelo logístico que usa só `estudante` como preditor concorda com essa leitura: o coeficiente sai positivo, 0,4049, e ser estudante soma à log-chance de inadimplência. O que acontece com esse coeficiente quando `saldo` e `renda` entram na mesma equação?

In [ ]:
X_multiplo = base[["saldo", "renda"]].copy()
X_multiplo["estudante_sim"] = (base["estudante"] == "sim").astype(int)

modelo_multiplo = LogisticRegression(
    C=np.inf, solver="newton-cholesky", tol=1e-8
).fit(X_multiplo, y)
coeficientes_multiplo = pd.Series(
    modelo_multiplo.coef_[0], index=modelo_multiplo.feature_names_in_
)
intercepto_multiplo = float(modelo_multiplo.intercept_[0])

(
    round(intercepto_multiplo, 4),
    round(float(coeficientes_multiplo["saldo"]), 6),
    f"{coeficientes_multiplo['renda']:.3e}",
    round(float(coeficientes_multiplo["estudante_sim"]), 4),
)

Com `saldo`, `renda` e `estudante` na mesma equação, o sinal se inverte: o coeficiente de estudante cai para -0,6468. É o mesmo tipo de reviravolta que a seção 8.3 viu no coeficiente de jornal, positivo sozinho e outra coisa depois que os preditores certos entram na equação.

In [ ]:
saldo_medio_estudante = float(estudantes["saldo"].mean())
saldo_medio_nao_estudante = float(nao_estudantes["saldo"].mean())

round(saldo_medio_estudante, 2), round(saldo_medio_nao_estudante, 2)

A explicação está no saldo. Em média, um estudante deve 987,82 dólares no cartão, e um não estudante, 771,77: 987,82 − 771,77 = 216,05 dólares a mais para o estudante.

Quanto pesa cada preditor? Os coeficientes crus não respondem, porque `saldo` e `renda` estão em dólares e `estudante_sim` é 0 ou 1: o coeficiente de `renda`, $3{,}033 \times 10^{-6}$ por dólar, seria mil vezes maior com `renda` em milhares, sem que nada mudasse no ajuste. Multiplicar cada coeficiente pelo desvio padrão do próprio preditor põe os três na mesma escala, a log-chance somada por um desvio padrão de variação:

In [ ]:
desvios = X_multiplo.std()
efeito_padronizado = coeficientes_multiplo * desvios

efeito_padronizado.round(4).to_dict(), efeito_padronizado.abs().idxmax()

> **🔧 Função**
>
> **`serie.abs().idxmax()`**: o rótulo do maior valor em módulo, aqui o nome do preditor de maior efeito padronizado.

Nessa escala, `saldo` pesa 2,7748, contra 0,0405 de `renda` e -0,2948 de `estudante_sim`, e `idxmax` confirma que é `saldo` o preditor de maior efeito. O estudante típico já carrega essa desvantagem, porque deve mais. O coeficiente isolado de `estudante` não separa os dois efeitos: sozinho, ele mistura o efeito de ser estudante com o efeito de, em média, dever mais.

In [ ]:
modelo_saldo_estudante = LogisticRegression(
    C=np.inf, solver="newton-cholesky", tol=1e-8
).fit(estudantes[["saldo"]], (estudantes["inadimplente"] == "sim").astype(int))
modelo_saldo_nao_estudante = LogisticRegression(
    C=np.inf, solver="newton-cholesky", tol=1e-8
).fit(nao_estudantes[["saldo"]], (nao_estudantes["inadimplente"] == "sim").astype(int))

grade_saldo = np.linspace(base["saldo"].min(), base["saldo"].max(), 300)
grade_saldo_df = pd.DataFrame({"saldo": grade_saldo})
prob_estudante = modelo_saldo_estudante.predict_proba(grade_saldo_df)[:, 1]
prob_nao_estudante = modelo_saldo_nao_estudante.predict_proba(grade_saldo_df)[:, 1]
estudante_sempre_abaixo = bool((prob_estudante < prob_nao_estudante).all())

faixa_comum_min = float(max(estudantes["saldo"].min(), nao_estudantes["saldo"].min()))
faixa_comum_max = float(min(estudantes["saldo"].max(), nao_estudantes["saldo"].max()))
grade_comum_df = pd.DataFrame(
    {"saldo": np.linspace(faixa_comum_min, faixa_comum_max, 300)}
)
prob_estudante_comum = modelo_saldo_estudante.predict_proba(grade_comum_df)[:, 1]
prob_nao_estudante_comum = modelo_saldo_nao_estudante.predict_proba(grade_comum_df)[
    :, 1
]
abaixo_na_faixa_comum = bool((prob_estudante_comum < prob_nao_estudante_comum).all())

(
    estudante_sempre_abaixo,
    round(faixa_comum_min, 2),
    round(faixa_comum_max, 2),
    abaixo_na_faixa_comum,
)

Dois modelos logísticos ajustados separadamente, um só com os estudantes e outro só com os não estudantes, cada um com `saldo` como único preditor, põem a curva do estudante abaixo da do não estudante em toda a grade: `estudante_sempre_abaixo` sai `True`. O mesmo vale restrito à faixa de saldo que os dois grupos de fato ocupam, de 0,00 a 2.499,02 dólares (`abaixo_na_faixa_comum`).

E sem modelo nenhum? Dividindo `saldo` em dez faixas com o mesmo número de clientes (os decis), dá para comparar a taxa observada dos dois grupos dentro de cada faixa:

In [ ]:
base_decis = base.copy()
base_decis["decil_saldo"] = pd.qcut(base_decis["saldo"], 10)
taxas_por_decil = (
    base_decis.groupby(["decil_saldo", "estudante"], observed=True)["inadimplente"]
    .apply(lambda s: (s == "sim").mean())
    .unstack("estudante")
)
ninguem_inadimplente = (taxas_por_decil["sim"] == 0) & (taxas_por_decil["não"] == 0)
estudante_abaixo = taxas_por_decil["sim"] < taxas_por_decil["não"]

inadimplentes_por_decil = base_decis.groupby("decil_saldo", observed=True)[
    "inadimplente"
].apply(lambda s: int((s == "sim").sum()))
decis_com_um_ou_dois = int(inadimplentes_por_decil.between(1, 2).sum())

(
    int(ninguem_inadimplente.sum()),
    int((estudante_abaixo & ~ninguem_inadimplente).sum()),
    int((~ninguem_inadimplente).sum()),
    decis_com_um_ou_dois,
)

> **🔧 Função**
>
> **`pd.qcut(serie, 10)`**: corta a série em 10 faixas com o mesmo número de observações cada, os decis. **`groupby([a, b])[coluna].apply(funcao)`**: aplica `funcao` à `coluna` dentro de cada combinação de `a` e `b`. **`.unstack(b)`** leva os valores de `b` para as colunas, uma coluna por categoria.

Em 4 dos 10 decis ninguém fica inadimplente, em nenhum dos grupos, e ali o dado não diz nada sobre quem é mais arriscado. Nos 6 decis restantes, a taxa observada do estudante fica abaixo da do não estudante em todos os 6. Dois desses 6, porém, têm só um ou dois inadimplentes (`decis_com_um_ou_dois`), e ali um ou dois casos a mais já mudariam a comparação; a evidência firme vem dos outros quatro.

In [ ]:
# Figura: Esquerda: taxa de inadimplência contra saldo. As curvas cheias são estimadas por um modelo logístico ajustado separadamente em cada grupo (laranja: estudante; azul: não estudante); as tracejadas marcam a taxa bruta observada de cada grupo, sem olhar para saldo. Direita: distribuição de saldo por grupo; o estudante se concentra em saldos mais altos.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.4))

ax1.plot(grade_saldo, prob_nao_estudante, color="C0", linewidth=2.2, label="não")
ax1.plot(grade_saldo, prob_estudante, color="C1", linewidth=2.2, label="sim")
ax1.axhline(taxa_nao_estudante, color="C0", linestyle="--", linewidth=1.2, alpha=0.7)
ax1.axhline(taxa_estudante, color="C1", linestyle="--", linewidth=1.2, alpha=0.7)
ax1.annotate(
    "taxa bruta: não",
    xy=(grade_saldo[-1], taxa_nao_estudante),
    xytext=(-104, -16),
    textcoords="offset points",
    fontsize=9,
    color="C0",
)
ax1.annotate(
    "taxa bruta: sim",
    xy=(grade_saldo[-1], taxa_estudante),
    xytext=(-104, 6),
    textcoords="offset points",
    fontsize=9,
    color="C1",
)
ax1.set_xlim(left=0)
ax1.set_ylim(-0.08, 1.02)
ax1.set_xlabel("saldo (dólares)")
ax1.set_ylabel("taxa de inadimplência")
ax1.legend(title="estudante", loc="upper left")

for dados, posicao, cor in [
    (nao_estudantes["saldo"], 1, "C0"),
    (estudantes["saldo"], 2, "C1"),
]:
    propriedades = dict(color=cor, linewidth=1.6)
    ax2.boxplot(
        dados,
        positions=[posicao],
        widths=0.5,
        boxprops=propriedades,
        whiskerprops=propriedades,
        capprops=propriedades,
        medianprops=propriedades,
        flierprops=dict(markeredgecolor=cor, markersize=3, alpha=0.5),
    )
ax2.set_xticks([1, 2])
ax2.set_xticklabels(["não", "sim"])
ax2.set_ylim(bottom=0)
ax2.set_xlabel("estudante")
ax2.set_ylabel("saldo (dólares)")

plt.tight_layout()
plt.show()

A curva da esquerda e os decis contam a mesma história. Com o saldo fixo, o estudante é o cliente menos arriscado: a curva laranja fica embaixo do primeiro ao último ponto da grade. As linhas tracejadas, que ignoram o saldo e mostram só a taxa bruta de cada grupo, invertem essa ordem. O estudante se concentra em saldos mais altos (os 216,05 dólares de diferença na média, medidos acima), e isso puxa a taxa geral dele para cima, mesmo com cada saldo individual sendo mais seguro para um estudante.

In [ ]:
prob_multiplo_todas = modelo_multiplo.predict_proba(X_multiplo)[:, 1]
mascara_estudante = X_multiplo["estudante_sim"] == 1

media_prob_estudante = float(prob_multiplo_todas[mascara_estudante].mean())
media_prob_nao_estudante = float(prob_multiplo_todas[~mascara_estudante].mean())

pd.DataFrame(
    {
        "prevista (média)": [media_prob_estudante, media_prob_nao_estudante],
        "bruta": [taxa_estudante, taxa_nao_estudante],
    },
    index=["estudante", "não estudante"],
).round(6)

A média da probabilidade que o modelo múltiplo prevê para cada estudante, sobre o saldo e a renda que ele de fato tem, é 0,043139; a mesma média entre não estudantes é 0,029195. A tabela põe as duas ao lado da taxa bruta de cada grupo, e elas coincidem até a sexta casa decimal. Não é acaso: sem penalização, a regressão logística com intercepto sempre reproduz a taxa observada em cada grupo de uma indicadora que está no modelo. O modelo múltiplo reproduz a taxa bruta de cada grupo e, ao mesmo tempo, diz que com o mesmo saldo o estudante é menos arriscado: as duas leituras são compatíveis.

## Logística Multinomial

> **📌 Nota**
>
> Esta seção corresponde à seção 4.3.5 de James et al. (2023).

`origem`, em `Auto`, tem três classes: americano, europeu e japonês. A curva em S da regressão logística devolve a probabilidade de uma classe contra a outra, e com três classes sempre sobra uma sem lugar nessa conta. Como dar a cada uma das três a sua probabilidade? É essa extensão, de duas classes para $K$, que a logística multinomial faz.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix

plt.style.use("estilo-figuras.mplstyle")

### A base, de novo

A saída é a mesma da seção 8.4: escolhe-se uma classe para servir de referência, a **base**, e cada uma das outras ganha o seu próprio conjunto de coeficientes, lido contra essa base. Com $K$ classes e uma base fixada, restam $K - 1$ conjuntos de coeficientes. Para preditores $x = (x_1, \ldots, x_p)$, a probabilidade de uma classe $k$ diferente da base, e a da própria base, ficam

$$
\Pr(Y = k \mid X = x) = \frac{e^{\beta_{k0} + \beta_{k1} x_1 + \cdots + \beta_{kp} x_p}}{1 + \sum_{l \neq \text{base}} e^{\beta_{l0} + \beta_{l1} x_1 + \cdots + \beta_{lp} x_p}},
$$

$$
\Pr(Y = \text{base} \mid X = x) = \frac{1}{1 + \sum_{l \neq \text{base}} e^{\beta_{l0} + \cdots + \beta_{lp} x_p}}.
$$

> **🔷 Conceito**
>
> Com $K$ categorias sem ordem natural, escolhe-se uma como base, e as $K - 1$ restantes ganham coeficientes próprios, lidos como log-chance contra essa base. É o mesmo gesto da seção 8.4, em que `drop_first` descartava uma categoria de um preditor e as indicadoras se liam contra ela; aqui, a categoria descartada é uma classe da resposta.

O `scikit-learn` ajusta uma forma equivalente, chamada *softmax*, que trata as $K$ classes de forma simétrica: nenhuma vira base, e todas ganham o próprio conjunto de coeficientes.

$$
\Pr(Y = k \mid X = x) = \frac{e^{\beta_{k0} + \beta_{k1} x_1 + \cdots + \beta_{kp} x_p}}{\sum_{l=1}^{K} e^{\beta_{l0} + \beta_{l1} x_1 + \cdots + \beta_{lp} x_p}}, \qquad k = 1, \ldots, K.
$$

Essa forma tem $K$ conjuntos de coeficientes, e não $K - 1$, e por isso sobram constantes livres: uma por coluna de coeficientes. Somar o mesmo número $c_j$ ao coeficiente $\beta_{kj}$ de todas as classes $k$ não muda probabilidade nenhuma, porque $c_j x_j$ aparece em todo expoente do numerador e do denominador e cancela na divisão. Isso vale separadamente para o intercepto e para cada preditor. A parametrização com base elimina essas sobras zerando de propósito a linha da classe escolhida. As duas formas dizem a mesma coisa, e o resto da seção mostra isso com o `Auto`.

### Três origens, cinco preditores

In [ ]:
auto = pd.read_csv("dados/Auto.csv")
n_total = len(auto)

auto["potencia"] = pd.to_numeric(auto["potencia"], errors="coerce")
n_com_interrogacao = int(auto["potencia"].isna().sum())

auto_limpo = auto.dropna(subset=["potencia"]).copy()
n_linhas = len(auto_limpo)

n_total, n_com_interrogacao, n_linhas

`dados/Auto.csv` tem 397 carros. `potencia` chega como texto, porque cinco linhas trazem `?` em vez de um número. `pd.to_numeric(auto["potencia"], errors="coerce")` converte a coluna para número e transforma esses cinco `?` em ausentes, e `dropna()` descarta as cinco linhas: sobram 392.

> **🔧 Função**
>
> **`pd.to_numeric(serie, errors="coerce")`**: converte uma coluna de texto em número; com `errors="coerce"`, o que não for número vira ausente (`NaN`) em vez de interromper a conversão.

In [ ]:
contagem_origem = auto_limpo["origem"].value_counts().sort_index()
contagem_origem

> **🔧 Função**
>
> **`serie.value_counts()`**: quantas vezes cada valor aparece. **`.sort_index()`** ordena o resultado pelo próprio valor (1, 2, 3) em vez de pela contagem.

Das 392 linhas, 245 são americanos (`origem` 1), 68 são europeus (`origem` 2) e 79 são japoneses (`origem` 3). 245 dos 392 carros (62,5%, isto é, 245/392) vêm dos Estados Unidos, e as outras duas origens dividem o resto. Cinco preditores técnicos (`cilindrada`, `potencia`, `peso`, `aceleracao` e `ano`) tentam prever a origem de cada carro.

### Uma linha de coeficientes por classe

In [ ]:
preditores = ["cilindrada", "potencia", "peso", "aceleracao", "ano"]
X = auto_limpo[preditores]
y = auto_limpo["origem"]

with warnings.catch_warnings(record=True) as avisos:
    warnings.simplefilter("always")
    modelo = LogisticRegression(C=np.inf, solver="newton-cholesky", tol=1e-8).fit(X, y)

n_avisos = len(avisos)
acuracia = float(modelo.score(X, y))

n_avisos, round(acuracia, 4)

> **🔧 Função**
>
> **`warnings.catch_warnings(record=True)`**: dentro do bloco `with`, guarda numa lista os avisos emitidos em vez de imprimi-los. Com `simplefilter("always")`, nenhum aviso é descartado, e `len(avisos)` conta quantos houve.

O ajuste usa a mesma `LogisticRegression` sem penalização da seção 9.2. Capturando todo aviso emitido durante o `fit`, `n_avisos` sai 0: nenhum aviso de convergência, nem de outro tipo. Sobre as mesmas 392 linhas que o treinaram, o modelo acerta 0,7883 das previsões. É a fração de acerto sobre o dado que ele já viu, e não o desempenho em dado novo; para isso seria preciso a taxa de erro de teste da seção 7.7, e o capítulo 10 mostra como estimá-la sem desperdiçar dado.

In [ ]:
tabela_coeficientes = pd.DataFrame(
    modelo.coef_,
    index=pd.Index(modelo.classes_, name="origem"),
    columns=modelo.feature_names_in_,
)
tabela_coeficientes.insert(0, "intercepto", modelo.intercept_)
tabela_coeficientes.round(4)

> **🔧 Função**
>
> **`modelo.classes_`**: as classes que o ajuste encontrou em `y`, em ordem. Cada linha de `coef_` e cada posição de `intercept_` correspondem a uma delas, na mesma ordem.
>
> **`df.insert(posicao, nome, valores)`**: acrescenta uma coluna na posição pedida, aqui a primeira. **`pd.Index(valores, name)`** monta os rótulos das linhas de uma tabela, e `name` dá nome ao próprio eixo, que aparece no canto da tabela (aqui, "origem").

`coef_` vem com três linhas, e não duas. `classes_` dá 1, 2 e 3, a mesma codificação de `origem`, e cada linha traz um coeficiente por preditor. Nenhuma das três é a base: é a parametrização softmax, que o `scikit-learn` ajusta sempre que o alvo tem mais de duas classes.

Qual desses números se lê como efeito? Nenhum sozinho. Como sobram essas constantes livres, a softmax sem penalização determina só as diferenças entre as linhas. O `scikit-learn` devolve a versão em que as três linhas somam zero, coluna a coluna:

In [ ]:
(
    bool(np.allclose(modelo.coef_.sum(axis=0), 0)),
    bool(np.isclose(modelo.intercept_.sum(), 0)),
)

> **🔧 Função**
>
> **`np.allclose(a, b)`** e **`np.isclose(a, b)`**: `True` se os valores coincidem até uma tolerância pequena, que absorve o último dígito de arredondamento. `allclose` responde por um array inteiro; `isclose`, elemento a elemento, ou para um número só.

Por isso o 13,8958 do intercepto europeu não tem significado próprio. O que se lê é a diferença entre duas linhas.

### A base muda, a previsão não

In [ ]:
classe_base = int(modelo.classes_[0])

coef_recentrado = modelo.coef_ - modelo.coef_[0]
intercepto_recentrado = modelo.intercept_ - modelo.intercept_[0]

tabela_recentrada = pd.DataFrame(
    coef_recentrado,
    index=pd.Index(modelo.classes_, name="origem"),
    columns=modelo.feature_names_in_,
)
tabela_recentrada.insert(0, "intercepto", intercepto_recentrado)
tabela_recentrada.round(4)

Subtrair a linha da origem 1 de toda linha de `coef_`, e o mesmo em `intercept_`, produz exatamente a forma com base do começo da seção: a linha da origem 1 zera, e as outras duas passam a ser lidas contra ela. O intercepto da origem 2, por exemplo, vira 19,9400, a diferença entre 13,8958 e -6,0442 da tabela anterior. Nada foi reajustado: a tabela só reorganiza os coeficientes do ajuste original. Mas a previsão continua a mesma?

In [ ]:
logitos_recentrados = X.values @ coef_recentrado.T + intercepto_recentrado
exp_logitos = np.exp(logitos_recentrados)
probabilidades_manuais = exp_logitos / exp_logitos.sum(axis=1, keepdims=True)

probabilidades_sklearn = modelo.predict_proba(X)
diferenca_maxima = float(np.abs(probabilidades_manuais - probabilidades_sklearn).max())
diferenca_abaixo_de_1e_12 = bool(diferenca_maxima < 1e-12)

previsao_manual = modelo.classes_[probabilidades_manuais.argmax(axis=1)]
previsao_sklearn = modelo.predict(X)
previsoes_batem = bool(np.array_equal(previsao_manual, previsao_sklearn))

diferenca_abaixo_de_1e_12, previsoes_batem

> **🔧 Função**
>
> **`A @ B.T`**: o produto de matrizes com a transposta de `B`. Aqui, multiplica cada carro (uma linha de `X`) por cada linha de coeficientes, e dá um logito por carro e por classe.
>
> **`arr.sum(axis=1, keepdims=True)`** soma cada linha e mantém o resultado como uma coluna, para que a divisão seja feita linha a linha. **`arr.argmax(axis=1)`** dá, em cada linha, a posição do maior valor.

A probabilidade recalculada à mão a partir desses logitos recentrados (a exponencial de cada logito dividida pela soma da linha, a própria definição de softmax) coincide com o `predict_proba` do ajuste original: a maior diferença, sobre as 392 × 3 entradas, fica abaixo de $10^{-12}$, e `diferenca_abaixo_de_1e_12` sai `True`. A previsão, a classe de maior probabilidade em cada linha, também bate: `previsoes_batem` sai `True`. Os coeficientes mudam de número; o que o modelo prevê para cada um dos 392 carros, não.

### Quais origens se confundem

Trocar a base não muda nada, e trocar a numeração das classes também não: o modelo multinomial não usa a ordem 1 < 2 < 3 em lugar nenhum. O problema da seção 9.1, uma escala inventada pela codificação, não existe aqui. O que resta perguntar é quais origens o modelo separa bem e quais confunde. A **matriz de confusão** responde: uma tabela que cruza a classe verdadeira com a classe prevista, uma célula por par.

In [ ]:
y_previsto = modelo.predict(X)
matriz = confusion_matrix(y, y_previsto, labels=modelo.classes_)

nomes = ["americano", "europeu", "japonês"]
matriz_rotulada = pd.DataFrame(
    matriz,
    index=pd.Index(nomes, name="origem verdadeira"),
    columns=pd.Index(nomes, name="origem prevista"),
)
matriz_rotulada

> **🔧 Função**
>
> **`confusion_matrix(y_verdadeiro, y_previsto, labels)`**: conta, para cada par de classes, quantas observações da classe da linha foram previstas como a classe da coluna. A diagonal são os acertos.
>
> - `labels=modelo.classes_`: fixa a ordem das linhas e colunas, para que ela não dependa de quais classes aparecem no dado.

A linha é a origem verdadeira e a coluna, a prevista; por isso a tabela, e a figura a seguir, rotulam os dois eixos. Contar os erros de cada par não basta, porque as origens têm tamanhos bem diferentes (245, 68 e 79 carros) e um par que reúne mais carros acumula mais erro só por isso. Dividindo os erros de cada par pelo número de carros que ele reúne:

In [ ]:
pares = {
    "americano–europeu": (0, 1),
    "europeu–japonês": (1, 2),
    "americano–japonês": (0, 2),
}
taxas_de_confusao = pd.Series(
    {
        par: (matriz[i, j] + matriz[j, i])
        / (contagem_origem.iloc[i] + contagem_origem.iloc[j])
        for par, (i, j) in pares.items()
    }
)

(taxas_de_confusao * 100).round(2).to_dict(), taxas_de_confusao.idxmax()

In [ ]:
# Figura: Matriz de confusão do ajuste sobre as 392 linhas de `Auto`, como bolhas: a área de cada bolha cresce com a raiz quadrada da contagem da célula, e o número exato está escrito dentro dela. Azul marca acerto (a diagonal); laranja marca erro. Eixo vertical: origem verdadeira; eixo horizontal: origem prevista.
fig, ax = plt.subplots(figsize=(6.2, 5.0))
for i in range(3):
    for j in range(3):
        contagem = int(matriz[i, j])
        cor = "C0" if i == j else "C1"
        ax.scatter([j], [i], s=70 * np.sqrt(contagem), color=cor, zorder=3)
        ax.annotate(
            str(contagem),
            xy=(j, i),
            ha="center",
            va="center",
            fontsize=9,
            color="white",
            zorder=4,
        )

ax.set_xticks([0, 1, 2])
ax.set_xticklabels(nomes)
ax.set_yticks([0, 1, 2])
ax.set_yticklabels(nomes)
ax.set_xlabel("origem prevista")
ax.set_ylabel("origem verdadeira")
ax.set_xlim(-0.6, 2.6)
ax.set_ylim(2.6, -0.6)
plt.tight_layout()
plt.show()

Entre americanos e europeus, o modelo confunde um com o outro em 5,43% dos carros do par; entre americanos e japoneses, em 10,19%; e entre europeus e japoneses, em 22,45%. `idxmax` confirma que o par mais confundido é europeu–japonês. Por quê? A média de cada preditor por origem dá uma pista:

In [ ]:
medias_por_origem = auto_limpo.groupby("origem")[preditores].mean().round(1)
medias_por_origem.index = nomes
medias_por_origem

In [ ]:
tamanho = ["cilindrada", "potencia", "peso"]
dist_europeu_japones = (
    medias_por_origem.loc["europeu", tamanho]
    - medias_por_origem.loc["japonês", tamanho]
).abs()
dist_europeu_americano = (
    medias_por_origem.loc["europeu", tamanho]
    - medias_por_origem.loc["americano", tamanho]
).abs()
europeu_mais_perto_do_japones = bool(
    (dist_europeu_japones < dist_europeu_americano).all()
)
europeu_mais_perto_do_japones

Os americanos têm, em média, cilindrada de 247,5 contra 109,6 dos europeus e 102,7 dos japoneses, e peso de 3.372,5 contra 2.433,5 e 2.221,2. Em cilindrada, potência e peso, `europeu_mais_perto_do_japones` confirma que a média europeia fica mais perto da japonesa do que da americana. É entre essas duas origens, parecidas no tamanho do carro, que o modelo mais confunde, em proporção dos carros do par.

## Modelos Generativos: LDA, QDA e Naive Bayes

> **📌 Nota**
>
> Esta seção corresponde às seções 4.4, 4.4.1, 4.4.2, 4.4.3 e 4.4.4 de James et al. (2023).

A regressão logística da seção 9.2 modela $\Pr(Y \mid X)$ direto, sem perguntar como os valores de $X$ se distribuem em cada classe. LDA, QDA e Naive Bayes fazem o caminho inverso: modelam como $X$ se distribui dentro de cada classe e só depois usam o teorema de Bayes para transformar essa distribuição em $\Pr(Y \mid X)$. A troca compensa em duas situações. Quando as classes estão bem separadas, os coeficientes da logística ficam instáveis. E quando há poucas observações e a distribuição de $X$ dentro de cada classe é aproximadamente normal, assumir essa forma rende uma estimativa mais precisa do que estimar a fronteira sem supor nada sobre a distribuição de $X$, como a logística faz.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from sklearn.discriminant_analysis import (
    LinearDiscriminantAnalysis,
    QuadraticDiscriminantAnalysis,
)
from sklearn.naive_bayes import GaussianNB

plt.style.use("estilo-figuras.mplstyle")

base = pd.read_csv("dados/Default.csv")
y = (base["inadimplente"] == "sim").astype(int)
X = base[["saldo", "renda"]].copy()
X["estudante_sim"] = (base["estudante"] == "sim").astype(int)

### Da distribuição de $X$ à probabilidade de $Y$

Cada um dos três métodos parte da mesma pergunta: como $X$ se distribui dentro da classe $k$? Chame de $\pi_k$ a proporção de observações que vêm da classe $k$ antes de olhar para $X$, a probabilidade *a priori*. Chame de $f_k(x)$ a densidade de $X$ entre as observações da classe $k$: grande onde um $X$ daquela classe é comum, pequena onde é raro. O teorema de Bayes transforma isso na probabilidade que interessa, a de $Y$ dado $X$:

$$
\Pr(Y = k \mid X = x) = \frac{\pi_k f_k(x)}{\sum_{l=1}^{K} \pi_l f_l(x)}.
$$

$\pi_k$ é fácil de estimar: é a fração das observações de treino que pertencem à classe $k$. $f_k(x)$ é a parte difícil, e é aí que LDA, QDA e Naive Bayes discordam. Cada um assume uma forma diferente para essa densidade, e é essa forma que decide o formato da fronteira entre as classes.

### LDA: a mesma covariância para todas as classes

A pergunta de partida é a mais simples possível: se $X$ tem, dentro de cada classe, a forma de sino da normal, com o mesmo espalhamento nas duas classes e só o centro mudando, onde fica a fronteira entre elas?

Com um preditor só, a LDA supõe que, dentro de cada classe $k$, $X$ segue uma normal com média $\mu_k$ própria da classe e uma variância $\sigma^2$ **comum a todas as classes**. Colocando essa densidade no teorema de Bayes e tomando o log, a classe escolhida é a que maximiza

$$
\delta_k(x) = x \cdot \frac{\mu_k}{\sigma^2} - \frac{\mu_k^2}{2\sigma^2} + \log \pi_k.
$$

Cada $\delta_k(x)$ é uma reta em $x$: inclinação $\mu_k / \sigma^2$, que cresce com a média da classe, mais um termo que só depende da classe. O ponto em que duas dessas retas se cruzam é a fronteira entre as duas classes. Com duas classes igualmente prováveis, é o ponto em que as duas densidades se cruzam:

In [ ]:
# Figura: 'Duas classes com um preditor só: normais com a mesma variância e médias -1,25 (azul) e 1,25 (laranja), igualmente prováveis. As curvas são $\pi_k f_k(x)$ de cada classe. A linha tracejada, onde elas se cruzam, é a fronteira de Bayes: à esquerda dela, a regra escolhe a classe azul; à direita, a laranja.'
def densidade_normal_1d(x, media, desvio):
    return np.exp(-0.5 * ((x - media) / desvio) ** 2) / (desvio * np.sqrt(2 * np.pi))


grade_1d = np.linspace(-4.5, 4.5, 400)
medias_1d = {"C0": -1.25, "C1": 1.25}
fronteira_1d = sum(medias_1d.values()) / 2

fig, ax = plt.subplots()
for cor, media in medias_1d.items():
    ax.plot(
        grade_1d,
        0.5 * densidade_normal_1d(grade_1d, media, 1.0),
        color=cor,
        linewidth=2.2,
    )
ax.axvline(fronteira_1d, color="#6C757D", linestyle="--", linewidth=1.4)
ax.set_ylim(bottom=0)
ax.set_xlabel("x")
ax.set_ylabel(r"$\pi_k f_k(x)$")
plt.tight_layout()
plt.show()

Com a variância comum e as duas classes igualmente prováveis, os dois sinos têm a mesma largura e a mesma altura, e a fronteira fica no meio do caminho entre as duas médias. É uma fronteira de um ponto só, a versão com um preditor de uma reta.

A figura usa $\mu_k$, $\sigma^2$ e $\pi_k$ verdadeiros, e por isso a linha tracejada é a fronteira de Bayes. A LDA não conhece esses valores: faz a mesma conta com a média de cada classe no treino, a variância combinada das classes e a fração de cada classe no treino, e a fronteira que ela acha é uma estimativa desta.

Com $p$ preditores, não é preciso acompanhar a álgebra matricial para ler o que importa. A média $\mu_k$ vira um vetor de $p$ médias, e a variância vira uma **matriz de covariância** $\Sigma$, uma tabela $p \times p$ com a variância de cada preditor na diagonal e a covariância de cada par fora dela. A regra tem a mesma forma do caso de um preditor, com a divisão por $\sigma^2$ trocada pela inversa de $\Sigma$:

$$
\delta_k(x) = x^T \Sigma^{-1} \mu_k - \frac{1}{2} \mu_k^T \Sigma^{-1} \mu_k + \log \pi_k.
$$

Cada $\delta_k(x)$ continua sendo um número, uma soma de termos lineares em $x_1, \ldots, x_p$, e a classe escolhida é a de maior $\delta_k(x)$.

> **🔷 Conceito**
>
> Com uma covariância só, o termo quadrático em $x$ que apareceria ao expandir a distância de $x$ até cada $\mu_k$ é o mesmo em toda classe, e cancela ao comparar $\delta_k(x)$ com $\delta_l(x)$. Sobra apenas uma combinação linear de $x$, e é daí que sai a fronteira **linear** que dá nome ao método.

### QDA: uma covariância por classe

Nem sempre as classes têm o mesmo espalhamento: uma nuvem pode ser alongada numa direção, e a outra, em outra. Para esse caso existe a QDA.

QDA solta a suposição mais forte da LDA: cada classe passa a ter a sua própria matriz de covariância, $\Sigma_k$. A classe escolhida passa a maximizar

$$
\delta_k(x) = -\frac{1}{2}(x-\mu_k)^T \Sigma_k^{-1} (x-\mu_k) - \frac{1}{2}\log|\Sigma_k| + \log \pi_k.
$$

Sem uma covariância comum, o termo quadrático em $x$ não cancela mais entre as classes, porque agora carrega o índice $k$. Sobra uma função **quadrática** de $x$, e a fronteira se curva.

A liberdade custa parâmetros: cada matriz de covariância com $p$ preditores tem $p(p+1)/2$ entradas livres a estimar, uma variância para cada preditor e uma covariância para cada par de preditores distintos. A LDA estima uma só; a QDA, uma por classe.

In [ ]:
p = X.shape[1]
K = int(y.nunique())

parametros_lda = p * (p + 1) // 2
parametros_qda = K * p * (p + 1) // 2
parametros_por_classe_com_10_preditores = 10 * 11 // 2

p, K, parametros_lda, parametros_qda, parametros_por_classe_com_10_preditores

Com os três preditores que esta seção usa sobre o `Default` (`saldo`, `renda` e o indicador de `estudante`, $p = 3$) e as duas classes de inadimplência ($K = 2$), a LDA estima 6 parâmetros de covariância, e a QDA, 12. A conta cresce depressa com $p$: com 10 preditores, seriam 55 por classe.

É a mesma troca que a seção 7.3 descreveu entre um ajuste paramétrico rígido e um mais livre, agora dentro da família normal. Menos parâmetros pedem menos dado e travam a fronteira numa forma rígida, linear. Mais parâmetros exigem mais dado e liberam uma fronteira mais flexível, quadrática. É o compromisso entre viés e variância da seção 7.6: a forma rígida da LDA tem menos variância e só carrega viés quando as classes não compartilham a covariância; a QDA não carrega esse viés, mas paga em variância.

### Naive Bayes: independência dentro da classe

Naive Bayes troca a suposição sobre a forma da densidade por uma suposição sobre a relação entre os preditores: dentro de cada classe, eles são **independentes**. A densidade conjunta se fatora num produto de densidades de um preditor só:

$$
f_k(x) = f_{k1}(x_1) \times f_{k2}(x_2) \times \cdots \times f_{kp}(x_p).
$$

Isso elimina a parte mais cara de estimar $f_k$: a associação entre os preditores, que com muitos preditores e pouco dado é a mais difícil de estimar bem. A suposição quase nunca é exata. Em `Default`, dentro de cada classe, `estudante_sim` e `renda` estão longe de independentes:

In [ ]:
correlacao_estudante_renda = {
    classe: round(
        float(X.loc[y == classe, "estudante_sim"].corr(X.loc[y == classe, "renda"])), 2
    )
    for classe in (0, 1)
}
correlacao_estudante_renda

A correlação é de -0,75 entre quem paga e de -0,79 entre quem fica inadimplente: estudantes têm renda bem menor, nos dois grupos. O Naive Bayes ignora essa associação. Mesmo assim, o método costuma funcionar bem: estimar a associação entre os preditores exige muito dado, e quando $n$ é pequeno diante de $p$, trocar essa estimativa pela suposição de que ela não existe tende a reduzir a variância mais do que aumenta o viés. Com poucos preditores e uma associação forte entre eles, a conta se inverte: a seção 9.6 mede esse caso.

> **🔷 Conceito**
>
> | | O que assume sobre $X$ dentro da classe | Fronteira |
> |---|---|---|
> | LDA | normal, uma covariância comum a todas as classes | linear |
> | QDA | normal, uma covariância própria por classe | quadrática |
> | Naive Bayes | preditores independentes dentro da classe | depende da densidade de cada preditor |

### Uma suposição que acerta, e uma que erra

A suposição da LDA, uma covariância só, acerta quando as classes de fato compartilham a forma da nuvem de pontos, e erra quando não compartilham. Quanto custa cada caso? Num dado simulado, quem gera o dado escolhe a covariância verdadeira de cada classe, e os dois casos se comparam lado a lado: duas classes com a mesma covariância num cenário, com covariâncias diferentes no outro. Um sorteio de treino só não decide a questão, porque num único sorteio pequeno a diferença entre LDA e QDA é tanto ruído quanto sinal. O que decide é repetir o sorteio muitas vezes contra o mesmo conjunto de teste.

In [ ]:
rng = np.random.default_rng(9)

media_azul = np.array([-1.5, -1.5])
media_laranja = np.array([1.5, 1.5])
cov_comum = np.array([[1.0, 0.7], [0.7, 1.0]])
cov_laranja_diferente = np.array([[1.0, -0.7], [-0.7, 1.0]])

n_treino_por_classe = 30
n_teste_por_classe = 20000
n_replicas = 200


def gera_duas_classes(cov_azul, cov_laranja, n, rng):
    azul = rng.multivariate_normal(media_azul, cov_azul, size=n)
    laranja = rng.multivariate_normal(media_laranja, cov_laranja, size=n)
    X_gerado = np.vstack([azul, laranja])
    y_gerado = np.concatenate([np.zeros(n), np.ones(n)])
    return X_gerado, y_gerado


# Um conjunto de teste grande e FIXO por painel, sorteado uma vez e nunca usado
# para ajustar nada: mede os n_replicas sorteios de treino a seguir contra o
# mesmo alvo, em vez de cada um contra o seu próprio.
X_teste_comum, y_teste_comum = gera_duas_classes(
    cov_comum, cov_comum, n_teste_por_classe, rng
)
X_teste_diferente, y_teste_diferente = gera_duas_classes(
    cov_comum, cov_laranja_diferente, n_teste_por_classe, rng
)

erros_lda_comum = np.empty(n_replicas)
erros_qda_comum = np.empty(n_replicas)
erros_lda_diferente = np.empty(n_replicas)
erros_qda_diferente = np.empty(n_replicas)

for i in range(n_replicas):
    X_treino_comum, y_treino_comum = gera_duas_classes(
        cov_comum, cov_comum, n_treino_por_classe, rng
    )
    X_treino_diferente, y_treino_diferente = gera_duas_classes(
        cov_comum, cov_laranja_diferente, n_treino_por_classe, rng
    )

    lda_comum_i = LinearDiscriminantAnalysis().fit(X_treino_comum, y_treino_comum)
    qda_comum_i = QuadraticDiscriminantAnalysis().fit(X_treino_comum, y_treino_comum)
    lda_diferente_i = LinearDiscriminantAnalysis().fit(
        X_treino_diferente, y_treino_diferente
    )
    qda_diferente_i = QuadraticDiscriminantAnalysis().fit(
        X_treino_diferente, y_treino_diferente
    )

    erros_lda_comum[i] = 1 - lda_comum_i.score(X_teste_comum, y_teste_comum)
    erros_qda_comum[i] = 1 - qda_comum_i.score(X_teste_comum, y_teste_comum)
    erros_lda_diferente[i] = 1 - lda_diferente_i.score(
        X_teste_diferente, y_teste_diferente
    )
    erros_qda_diferente[i] = 1 - qda_diferente_i.score(
        X_teste_diferente, y_teste_diferente
    )

media_erro_lda_comum = float(erros_lda_comum.mean())
media_erro_qda_comum = float(erros_qda_comum.mean())
media_erro_lda_diferente = float(erros_lda_diferente.mean())
media_erro_qda_diferente = float(erros_qda_diferente.mean())

fracao_lda_vence_comum = float((erros_lda_comum < erros_qda_comum).mean())
fracao_qda_vence_diferente = float((erros_qda_diferente < erros_lda_diferente).mean())

(
    round(media_erro_lda_comum, 4),
    round(media_erro_qda_comum, 4),
    round(fracao_lda_vence_comum, 2),
    round(media_erro_lda_diferente, 4),
    round(media_erro_qda_diferente, 4),
    round(fracao_qda_vence_diferente, 2),
)

> **🔧 Função**
>
> **`np.empty(n)`**: um array de `n` posições ainda sem valores, preenchido dentro do laço. **`rng.multivariate_normal(media, cov, size)`**: sorteia `size` pontos de uma normal com várias coordenadas, com vetor de médias `media` e matriz de covariância `cov`. **`np.vstack`** empilha as duas nuvens de pontos uma sobre a outra, e **`np.concatenate`** junta os dois vetores de rótulos.
>
> **`LinearDiscriminantAnalysis().fit(X, y)`** e **`QuadraticDiscriminantAnalysis().fit(X, y)`**: ajustam LDA e QDA, estimando $\pi_k$, $\mu_k$ e a covariância (uma só ou uma por classe) a partir do treino. **`modelo.score(X, y)`**, num classificador, é a fração de acertos; `1 - score` é a taxa de erro.

Repetindo o sorteio de treino 200 vezes, sempre com 30 pontos por classe, contra o mesmo conjunto de teste de 20.000 pontos por classe em cada cenário: com a covariância comum, a LDA erra em média 5,56% contra 5,72% da QDA, e erra menos em 73% dos 200 sorteios, uma vantagem sistemática mas modesta. Com covariâncias diferentes, a LDA erra em média 2,66% contra 1,05% da QDA, e a QDA erra menos em 99% dos 200 sorteios. Neste cenário, com dois preditores e 30 pontos por classe, a suposição da LDA compensa pouco quando vale, e custa caro quando não vale.

O quadro não é geral. A QDA estima uma covariância inteira por classe, e o número de parâmetros cresce com $p(p+1)/2$: com muitos preditores e pouco dado, a variância da QDA pesa mais, e aí é a LDA que leva vantagem sempre que a suposição dela está perto de valer. Com dois preditores, essa conta ainda é pequena.

Com um treino bem maior, 1.000 pontos por classe em vez de 30, as fronteiras ajustadas ficam perto da verdade em vez de perto do ruído de uma amostra pequena. É essa fronteira que a figura a seguir desenha, ao lado da **fronteira de Bayes**: a fronteira ótima, que sai direto das densidades normais verdadeiras que geraram o dado, sem ajustar modelo nenhum. Como as duas classes são igualmente prováveis, o termo $\log \pi_k$ é o mesmo nas duas e cancela, e a fronteira de Bayes fica onde as duas densidades se igualam.

In [ ]:
# Figura: Duas classes simuladas, azul e laranja, com 1.000 pontos de treino cada e a mesma separação entre médias nos dois painéis. A linha cinza pontilhada é a fronteira de Bayes, calculada das densidades normais verdadeiras; a verde sólida, a fronteira da LDA; a roxa tracejada, a da QDA. Esquerda: as duas classes compartilham a mesma matriz de covariância (correlação 0,7). Direita: a laranja tem correlação -0,7 em vez de 0,7.
n_treino_figura_por_classe = 1000

X_treino_comum_fig, y_treino_comum_fig = gera_duas_classes(
    cov_comum, cov_comum, n_treino_figura_por_classe, rng
)
X_treino_diferente_fig, y_treino_diferente_fig = gera_duas_classes(
    cov_comum, cov_laranja_diferente, n_treino_figura_por_classe, rng
)

lda_comum_fig = LinearDiscriminantAnalysis().fit(X_treino_comum_fig, y_treino_comum_fig)
qda_comum_fig = QuadraticDiscriminantAnalysis().fit(
    X_treino_comum_fig, y_treino_comum_fig
)
lda_diferente_fig = LinearDiscriminantAnalysis().fit(
    X_treino_diferente_fig, y_treino_diferente_fig
)
qda_diferente_fig = QuadraticDiscriminantAnalysis().fit(
    X_treino_diferente_fig, y_treino_diferente_fig
)


def log_densidade_normal(pontos, media, cov):
    p = media.shape[0]
    inv_cov = np.linalg.inv(cov)
    log_det_cov = np.log(np.linalg.det(cov))
    diff = pontos - media
    forma_quadratica = np.einsum("ij,jk,ik->i", diff, inv_cov, diff)
    return -0.5 * p * np.log(2 * np.pi) - 0.5 * log_det_cov - 0.5 * forma_quadratica


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.6), sharex=True, sharey=True)

grade_x = np.linspace(-4.5, 4.5, 300)
grade_y = np.linspace(-4.5, 4.5, 300)
malha_x, malha_y = np.meshgrid(grade_x, grade_y)
pontos_grade = np.column_stack([malha_x.ravel(), malha_y.ravel()])

paineis = [
    (
        ax1,
        X_treino_comum_fig,
        y_treino_comum_fig,
        lda_comum_fig,
        qda_comum_fig,
        cov_comum,
        "mesma covariância",
    ),
    (
        ax2,
        X_treino_diferente_fig,
        y_treino_diferente_fig,
        lda_diferente_fig,
        qda_diferente_fig,
        cov_laranja_diferente,
        "covariâncias diferentes",
    ),
]
for (
    ax,
    X_treino,
    y_treino,
    modelo_lda,
    modelo_qda,
    cov_laranja_verdadeira,
    titulo,
) in paineis:
    previsao_lda_grade = modelo_lda.predict(pontos_grade).reshape(malha_x.shape)
    previsao_qda_grade = modelo_qda.predict(pontos_grade).reshape(malha_x.shape)
    diferenca_log_densidade = (
        log_densidade_normal(pontos_grade, media_azul, cov_comum)
        - log_densidade_normal(pontos_grade, media_laranja, cov_laranja_verdadeira)
    ).reshape(malha_x.shape)

    ax.scatter(
        X_treino[y_treino == 0, 0],
        X_treino[y_treino == 0, 1],
        color="C0",
        s=6,
        zorder=3,
    )
    ax.scatter(
        X_treino[y_treino == 1, 0],
        X_treino[y_treino == 1, 1],
        color="C1",
        s=6,
        zorder=3,
    )
    ax.contour(
        malha_x,
        malha_y,
        diferenca_log_densidade,
        levels=[0],
        colors="#6C757D",
        linewidths=2.0,
        linestyles="dotted",
        zorder=4,
    )
    ax.contour(
        malha_x, malha_y, previsao_lda_grade, levels=[0.5], colors="C2", linewidths=2.2
    )
    ax.contour(
        malha_x,
        malha_y,
        previsao_qda_grade,
        levels=[0.5],
        colors="C3",
        linewidths=2.2,
        linestyles="dashed",
    )
    ax.set_xlabel("x1")
    ax.set_title(titulo)

ax1.set_ylabel("x2")

legenda_fronteiras = [
    Line2D(
        [],
        [],
        marker="o",
        linestyle="None",
        color="C0",
        markersize=7,
        label="classe azul",
    ),
    Line2D(
        [],
        [],
        marker="o",
        linestyle="None",
        color="C1",
        markersize=7,
        label="classe laranja",
    ),
    Line2D(
        [],
        [],
        color="#6C757D",
        linewidth=1.8,
        linestyle=":",
        label="fronteira de Bayes",
    ),
    Line2D([], [], color="C2", linewidth=2.2, label="fronteira LDA"),
    Line2D(
        [], [], color="C3", linewidth=2.2, linestyle="dashed", label="fronteira QDA"
    ),
]
fig.legend(
    handles=legenda_fronteiras,
    loc="lower center",
    ncols=5,
    bbox_to_anchor=(0.5, -0.05),
    frameon=False,
)

plt.tight_layout()
plt.show()

> **🔧 Função**
>
> **`ax.contour(x, y, z, levels=[0])`**: com um nível só, desenha a curva em que `z` vale exatamente 0. Aqui, `z` é a diferença entre as log-densidades das duas classes (a fronteira de Bayes) ou a classe prevista em cada ponto da grade (com `levels=[0.5]`, a fronteira ajustada).
>
> **`np.linalg.inv(M)`** dá a inversa de `M`, e **`np.linalg.det(M)`** dá o determinante de `M`, que entra no termo $\log|\Sigma_k|$ da densidade normal. **`np.einsum("ij,jk,ik->i", d, M, d)`** calcula, para cada linha $d_i$ de `d`, o número $d_i^T M d_i$, a parte quadrática da densidade normal. **`fig.legend(handles=...)`** monta uma legenda única para a figura inteira a partir de marcadores avulsos criados com **`Line2D`**.

A fronteira de Bayes serve de régua para julgar as outras duas. No painel de covariância comum, ela é reta, porque as duas classes de fato compartilham $\Sigma$, e a fronteira da LDA praticamente a cobre; a da QDA se afasta dela nas pontas, curvando-se para acompanhar uma diferença de covariância que não existe. No painel de covariâncias diferentes, a fronteira de Bayes se curva em volta do canto superior da nuvem azul, e é a da QDA que a acompanha de perto; a da LDA, presa a ser reta, não consegue seguir a curva. No canto de cima à direita, a fronteira de Bayes tem ainda um segundo trecho: com covariâncias de sinais opostos, ela é uma hipérbole, com dois ramos. Longe das duas médias, na direção da diagonal em que a nuvem azul é alongada, a classe azul volta a ser a mais provável. A QDA reproduz também esse segundo ramo, o que a LDA não teria como fazer. É a mesma assimetria que a média sobre 200 sorteios mediu acima, agora visível.

### Os três sobre o `Default`

Sobre o `Default`, com os mesmos três preditores (`saldo`, `renda` e o indicador de `estudante`), as classes `LinearDiscriminantAnalysis`, `QuadraticDiscriminantAnalysis` e `GaussianNB` do `scikit-learn` ajustam LDA, QDA e Naive Bayes como definidos acima. A `GaussianNB` estima cada densidade $f_{kj}$ de um preditor só como uma normal, a opção mais simples para um preditor quantitativo. O indicador de estudante, que só vale 0 ou 1, também recebe uma normal: uma aproximação grosseira, porque a `GaussianNB` não distingue tipos de preditor. Para um preditor qualitativo, o natural seria contar a proporção de cada valor dentro da classe, e é o que a `CategoricalNB` faz. LDA e QDA fazem a mesma aproximação: supõem que os três preditores, o indicador inclusive, seguem juntos uma normal dentro de cada classe.

In [ ]:
lda = LinearDiscriminantAnalysis().fit(X, y)
qda = QuadraticDiscriminantAnalysis().fit(X, y)
nb = GaussianNB().fit(X, y)

erro_lda = float((lda.predict(X) != y).mean())
erro_qda = float((qda.predict(X) != y).mean())
erro_nb = float((nb.predict(X) != y).mean())
erro_trivial = float(y.mean())

round(erro_lda, 4), round(erro_qda, 4), round(erro_nb, 4), round(erro_trivial, 4)

> **🔧 Função**
>
> **`GaussianNB().fit(X, y)`**: ajusta o Naive Bayes com uma normal para cada preditor dentro de cada classe. Como LDA e QDA, prevê com `.predict(X)` e dá probabilidades com `.predict_proba(X)`.

In [ ]:
pd.DataFrame(
    {
        "erro sobre as próprias linhas do ajuste": [
            erro_lda,
            erro_qda,
            erro_nb,
            erro_trivial,
        ]
    },
    index=["LDA", "QDA", "Naive Bayes", 'sempre "não"'],
).round(4)

In [ ]:
qda_erra_menos_que_lda = bool(erro_qda < erro_lda)
lda_erra_menos_que_nb = bool(erro_lda < erro_nb)
diferenca_lda_qda = round(abs(erro_lda - erro_qda), 4)
todos_abaixo_do_trivial = bool(max(erro_lda, erro_qda, erro_nb) < erro_trivial)

(
    qda_erra_menos_que_lda,
    lda_erra_menos_que_nb,
    diferenca_lda_qda,
    todos_abaixo_do_trivial,
)

Sobre as mesmas linhas que ajustaram os modelos, a QDA erra menos (2,70%), e `qda_erra_menos_que_lda` confirma, mas por uma margem mínima sobre a LDA (2,76%): a diferença é de 0,0006, perto demais para chamar de vitória. A LDA, por sua vez, erra menos que o Naive Bayes (2,93%), e `lda_erra_menos_que_nb` confirma. Prever `não` para todo mundo, sem olhar para `saldo`, `renda` ou `estudante`, erra 3,33%. Os três classificadores ficam abaixo disso (`todos_abaixo_do_trivial`), mas só um pouco. Essa proximidade não fecha a questão, porque o erro total esconde *como* se erra: quantos inadimplentes cada método deixa passar, e quantos bons pagadores acusa. Medir esse *como* é o assunto da seção 9.5.

## Avaliando um Classificador

> **📌 Nota**
>
> Esta seção corresponde à seção 4.4.2 de James et al. (2023).

Um classificador de inadimplência pode errar de dois jeitos bem diferentes: dizer `não` para quem de fato fica inadimplente, e dizer `sim` para quem não fica. O erro total soma os dois, como se custassem o mesmo. Esta seção separa um do outro, com o classificador logístico da seção 9.2 (`saldo`, `renda` e o indicador de `estudante`) como caso de trabalho. Tudo o que segue é medido sobre as mesmas linhas que ajustaram o modelo, e não em dado novo. O vocabulário que sai daqui vale para julgar qualquer classificador.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.discriminant_analysis import (
    LinearDiscriminantAnalysis,
    QuadraticDiscriminantAnalysis,
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.naive_bayes import GaussianNB

plt.style.use("estilo-figuras.mplstyle")

base = pd.read_csv("dados/Default.csv")
y = (base["inadimplente"] == "sim").astype(int)
X = base[["saldo", "renda"]].copy()
X["estudante_sim"] = (base["estudante"] == "sim").astype(int)

modelo = LogisticRegression(C=np.inf, solver="newton-cholesky", tol=1e-8).fit(X, y)
proba = modelo.predict_proba(X)[:, 1]

### A acurácia engana quando a classe é rara

In [ ]:
n_total = len(y)
acerto_trivial = float((y == 0).mean())
erro_trivial = float(y.mean())

n_total, round(acerto_trivial, 4), round(erro_trivial, 4)

Um classificador que responde sempre `não`, sem olhar `saldo`, `renda` nem `estudante`, acerta 96,67% das 10.000 linhas, porque 96,67% delas de fato não são inadimplentes; ele erra nos 3,33% restantes, e só neles. Esse número não mede o classificador: mede a proporção da classe rara. Um classificador de verdade precisa ser julgado contra ele, e não contra 100%, e a acurácia sozinha não distingue um classificador que aprendeu algo de um que só repete a classe mais comum.

### A matriz de confusão

In [ ]:
y_previsto_05 = modelo.predict(X)
matriz_05 = confusion_matrix(y, y_previsto_05, labels=[0, 1])

matriz_05_rotulada = pd.DataFrame(
    matriz_05,
    index=pd.Index(["não", "sim"], name="inadimplente verdadeiro"),
    columns=pd.Index(["não", "sim"], name="inadimplente previsto"),
)
matriz_05_rotulada

`modelo.predict(X)` usa o limiar padrão de 0,5: prevê `sim` quando a probabilidade ajustada passa de 0,5, e `não` caso contrário. A matriz segue a convenção da seção 9.3, linha para a verdade e coluna para a previsão. Chame as quatro células de VN (verdadeiro negativo), FP (falso positivo), FN (falso negativo) e VP (verdadeiro positivo), na ordem em que aparecem na tabela: linha `não`, colunas `não` e `sim`; depois linha `sim`, colunas `não` e `sim`.

In [ ]:
vn_05, fp_05 = int(matriz_05[0, 0]), int(matriz_05[0, 1])
fn_05, vp_05 = int(matriz_05[1, 0]), int(matriz_05[1, 1])

erro_05 = (fp_05 + fn_05) / len(y)
precisao_05 = vp_05 / (vp_05 + fp_05)
revocacao_05 = vp_05 / (vp_05 + fn_05)

(
    vn_05,
    fp_05,
    fn_05,
    vp_05,
    round(erro_05, 4),
    round(precisao_05, 4),
    round(revocacao_05, 4),
)

No limiar padrão, são 9.627 acertos entre quem não fica inadimplente, contra 40 falsos alarmes, e 105 acertos entre quem fica, contra 228 que passam despercebidos. O erro total, (FP + FN) dividido pelas 10.000 linhas, é 0,0268, abaixo dos 0,0333 do classificador trivial. Mas FP + FN mistura dois erros de custo bem diferente, e é aí que entram a precisão e a revocação:

$$
\text{precisão} = \frac{VP}{VP + FP}, \qquad \text{revocação} = \frac{VP}{VP + FN}.
$$

A precisão responde: entre quem o modelo acusou de inadimplente, quantos de fato eram? Aqui, 105 entre 105 + 40 = 145, ou 0,7241. A revocação responde: entre quem de fato ficou inadimplente, quantos o modelo achou? São 105 entre 105 + 228 = 333, ou 0,3153. O erro total de 2,68% parece ótimo até se perguntar de quem é o erro: no limiar padrão, o modelo acha 105 dos 333 inadimplentes e deixa 228 passarem sem sinal nenhum.

> **🔷 Conceito**
>
> Matriz de confusão, precisão e revocação são o vocabulário mínimo para julgar um classificador. A matriz separa os quatro jeitos de acertar e errar: VN, FP, FN e VP. A **precisão**, VP / (VP + FP), mede a confiança de um alarme: quando o modelo diz `sim`, com que frequência está certo. A **revocação**, VP / (VP + FN), mede a cobertura sobre quem de fato é positivo: da classe que interessa achar, quanto o modelo encontra. Nenhuma das duas depende do algoritmo que gerou a previsão; servem do mesmo jeito para regressão logística, LDA, QDA, Naive Bayes ou qualquer outro classificador.

### O limiar é escolha, não verdade

`predict` decide com um limiar fixo, 0,5, mas nada obriga a esse número: a mesma probabilidade ajustada, comparada com um limiar mais baixo, muda quem entra em cada célula da matriz. Baixando o limiar para 0,2, o que deve acontecer com os falsos alarmes e com os inadimplentes perdidos?

In [ ]:
def metricas(limiar):
    previsto = (proba >= limiar).astype(int)
    matriz = confusion_matrix(y, previsto, labels=[0, 1])
    vn, fp = int(matriz[0, 0]), int(matriz[0, 1])
    fn, vp = int(matriz[1, 0]), int(matriz[1, 1])
    erro = (fp + fn) / len(y)
    precisao = vp / (vp + fp) if (vp + fp) else float("nan")
    revocacao = vp / (vp + fn) if (vp + fn) else float("nan")
    return vn, fp, fn, vp, erro, precisao, revocacao


limiares_nomeados = [0.5, 0.2, 0.1]
resultados = [metricas(limiar) for limiar in limiares_nomeados]

tabela_limiares = pd.DataFrame(
    {
        "VN": [r[0] for r in resultados],
        "FP": [r[1] for r in resultados],
        "FN": [r[2] for r in resultados],
        "VP": [r[3] for r in resultados],
        "erro": [r[4] for r in resultados],
        "precisão": [r[5] for r in resultados],
        "revocação": [r[6] for r in resultados],
    },
    index=pd.Index(limiares_nomeados, name="limiar"),
)
tabela_limiares.round(4)

Baixar o limiar de 0,5 para 0,2 muda a matriz inteira: FP sobe de 40 para 277, FN cai de 228 para 130, e VP sobe de 105 para 203. A revocação vai de 0,3153 para 0,6096, quase o dobro de inadimplentes encontrados, e a precisão cai de 0,7241 para 0,4229, porque boa parte de quem passa a ser acusado não é inadimplente. O erro total também sobe, de 0,0268 para 0,0407. Em 0,1, a revocação chega a 0,7447 e a precisão cai a 0,3069, com erro total de 0,0645. Nenhum dos três limiares da tabela melhora a revocação e a precisão ao mesmo tempo: cada um compra uma pagando com a outra.

Três limiares escolhidos à mão mostram a direção da troca, mas não dizem o que acontece no intervalo entre eles. A varredura a seguir acompanha os dois tipos de erro, os inadimplentes perdidos e os alarmes falsos, com o limiar indo de 0,0 a 0,5 em passos de 0,002:

In [ ]:
n_pos = int(y.sum())
n_neg = int((y == 0).sum())

grade_limiares = np.linspace(0.0, 0.5, 251)
erro_total = np.empty_like(grade_limiares)
taxa_perdidos = np.empty_like(grade_limiares)
taxa_falso_alarme = np.empty_like(grade_limiares)
fp_contagem = np.empty(len(grade_limiares), dtype=int)
fn_contagem = np.empty(len(grade_limiares), dtype=int)

for i, limiar in enumerate(grade_limiares):
    previsto = (proba >= limiar).astype(int)
    fp = int(((y == 0) & (previsto == 1)).sum())
    fn = int(((y == 1) & (previsto == 0)).sum())
    fp_contagem[i] = fp
    fn_contagem[i] = fn
    erro_total[i] = (fp + fn) / len(y)
    taxa_perdidos[i] = fn / n_pos
    taxa_falso_alarme[i] = fp / n_neg

perdidos_nunca_cai_com_limiar = bool((np.diff(taxa_perdidos) >= 0).all())
falso_alarme_nunca_sobe_com_limiar = bool((np.diff(taxa_falso_alarme) <= 0).all())

# Onde as duas TAXAS (denominadores 333 e 9.667) mais se aproximam.
indice_cruzamento_taxas = int(np.argmin(np.abs(taxa_perdidos - taxa_falso_alarme)))
limiar_cruzamento_taxas = float(grade_limiares[indice_cruzamento_taxas])

# Onde as duas CONTAGENS (mesma unidade: pessoas) mais se aproximam: um
# ponto diferente do cruzamento das taxas, de propósito.
indice_cruzamento_contagem = int(np.argmin(np.abs(fp_contagem - fn_contagem)))
limiar_cruzamento_contagem = float(grade_limiares[indice_cruzamento_contagem])

erro_minimo = float(erro_total.min())
limiares_no_minimo = [
    round(float(t), 3) for t in grade_limiares[erro_total == erro_total.min()]
]

(
    perdidos_nunca_cai_com_limiar,
    falso_alarme_nunca_sobe_com_limiar,
    round(limiar_cruzamento_taxas, 3),
    int(fp_contagem[indice_cruzamento_taxas]),
    int(fn_contagem[indice_cruzamento_taxas]),
    round(limiar_cruzamento_contagem, 3),
    int(fp_contagem[indice_cruzamento_contagem]),
    int(fn_contagem[indice_cruzamento_contagem]),
    round(erro_minimo, 4),
    limiares_no_minimo,
    round(float(erro_total[-1]), 4),
)

> **🔧 Função**
>
> **`np.empty_like(arr)`**: um array do mesmo formato de `arr`, ainda sem valores definidos, para ser preenchido dentro do laço.

Nos 251 limiares varridos, a fração de inadimplentes perdidos nunca cai quando o limiar sobe (`perdidos_nunca_cai_com_limiar` é `True`), e a fração de alarmes falsos entre adimplentes nunca sobe (`falso_alarme_nunca_sobe_com_limiar` é `True`). As duas nunca andam no mesmo sentido, e não só nos três pontos da tabela.

As duas *taxas* se cruzam perto do limiar 0,038, mas são taxas com bases diferentes: 333 inadimplentes contra 9.667 adimplentes. Nesse mesmo limiar, em número de pessoas, o modelo produz 1.179 falsos alarmes contra só 40 inadimplentes perdidos, quase 30 alarmes falsos por inadimplente não achado. Taxas iguais não são erros iguais: quem opera o cartão sente a contagem, e não a taxa. As duas *contagens* só se equilibram bem mais adiante, perto do limiar 0,278, com 160 falsos alarmes contra 159 perdidos.

O menor erro total da varredura, 0,0261, aparece em dois limiares, 0,428 e 0,430 (`limiares_no_minimo`). É um empate exato, porque `erro_total` é uma contagem inteira dividida por 10.000. Não fica exatamente em 0,5, que dá 0,0268, mas perto de 0,5 o erro total já está quase plano.

In [ ]:
# Figura: Taxas de erro contra o limiar de decisão, de 0,0 a 0,5, para o classificador logístico do Default. Verde: erro total. Laranja tracejada: fração de inadimplentes (verdadeiros \"sim\") que o modelo deixa passar. Azul pontilhada: fração de adimplentes (verdadeiros \"não\") que o modelo alarma por engano. As duas últimas se cruzam perto de 0,04 e se movem em direções opostas em toda a faixa. O erro total é a média das outras duas, com peso de 96,67% para os alarmes falsos e 3,33% para os perdidos; por isso a curva verde acompanha a azul de perto nos limiares baixos e passa pelo mesmo ponto de cruzamento.
fig, ax = plt.subplots()
ax.plot(grade_limiares, erro_total, color="C2", linewidth=2.2, label="erro total")
ax.plot(
    grade_limiares,
    taxa_perdidos,
    color="C1",
    linewidth=2.2,
    linestyle="--",
    label="inadimplentes perdidos",
)
ax.plot(
    grade_limiares,
    taxa_falso_alarme,
    color="C0",
    linewidth=2.2,
    linestyle=":",
    label="alarme falso entre adimplentes",
)
ax.scatter(
    [limiar_cruzamento_taxas],
    [taxa_perdidos[indice_cruzamento_taxas]],
    color="#6C757D",
    zorder=5,
    s=28,
)
ax.annotate(
    f"limiar ≈ {limiar_cruzamento_taxas:.3f}".replace(".", ","),
    xy=(limiar_cruzamento_taxas, taxa_perdidos[indice_cruzamento_taxas]),
    xytext=(0.25, 0.25),
    textcoords="data",
    fontsize=9,
    arrowprops=dict(arrowstyle="-", color="#6C757D", linewidth=0.8),
)
ax.set_xlim(0, 0.5)
ax.set_ylim(0, 1)
ax.set_xlabel("limiar de decisão")
ax.set_ylabel("taxa")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

O que decide qual limiar usar não sai dessa figura. Baixar o limiar nunca acha menos inadimplentes, e nunca gera menos alarmes falsos, e nenhum ponto do gráfico é "o melhor" sem responder a uma pergunta que a figura não faz: quanto custa, para quem opera o cartão, deixar passar um inadimplente comparado a incomodar um cliente em dia? Essa é uma pergunta sobre o problema, e não sobre estatística. A figura não escolhe um limiar; mostra o que cada escolha custa.

### A curva ROC

A varredura acima olha limiar por limiar. A **curva ROC** resume todos eles de uma vez: para cada limiar possível, marca a taxa de acerto entre inadimplentes (a revocação) contra a taxa de alarme falso entre adimplentes, sem mostrar o limiar em si.

In [ ]:
fpr, tpr, limiares_roc = roc_curve(y, proba)
auc = roc_auc_score(y, proba)

n_proba_distintas = len(np.unique(proba))
n_pares_saldo_renda_repetidos = int(base.duplicated(["saldo", "renda"]).sum())
limiar_roc_inicial = float(limiares_roc[0])
curva_nunca_abaixo_da_diagonal = bool((tpr >= fpr).all())

(
    n_proba_distintas,
    n_pares_saldo_renda_repetidos,
    len(fpr),
    limiar_roc_inicial,
    round(float(auc), 4),
    curva_nunca_abaixo_da_diagonal,
)

> **🔧 Função**
>
> **`roc_curve(y, probabilidades)`**: devolve três arrays alinhados, a taxa de falso alarme (`fpr`), a revocação (`tpr`) e o limiar de cada ponto da curva. **`roc_auc_score(y, probabilidades)`** dá a área sob essa curva. As duas recebem as probabilidades, e não as previsões 0/1, porque percorrem todos os limiares.
>
> **`np.unique(arr)`**: os valores distintos de `arr`, sem repetição.
>
> **`df.duplicated(colunas)`**: marca com `True` cada linha que repete, nas `colunas` dadas, uma linha anterior; `.sum()` conta as repetições.

`roc_curve` não devolve um ponto por probabilidade observada. As probabilidades ajustadas têm 10.000 valores distintos (`n_proba_distintas`), um por cliente; e nenhum par de clientes repete ao mesmo tempo saldo e renda (`n_pares_saldo_renda_repetidos` sai 0). Por padrão, a função descarta os pontos que ficam em linha reta com os vizinhos e não mudam a forma da curva (`drop_intermediate=True`), e acrescenta no começo um limiar artificial, `inf`, que não é probabilidade nenhuma: daí 440 pontos, bem menos que um por cliente. São os vértices que bastam para desenhar a curva inteira.

Em nenhum dos 440 a revocação fica abaixo da taxa de falso alarme (`curva_nunca_abaixo_da_diagonal` é `True`): a curva não passa para baixo da diagonal em ponto nenhum. `roc_auc_score` resume a curva inteira num número, a área sob ela, **AUC**: 0,9496, perto do máximo de 1,0 e bem acima dos 0,5 de um classificador sem informação nenhuma.

In [ ]:
# Figura: Curva ROC do classificador logístico sobre o `Default` (azul): revocação (TPR) contra taxa de falso alarme (FPR), para os 440 pontos que definem a curva. A diagonal pontilhada cinza marca um classificador sem informação nenhuma, com FPR e revocação crescendo juntos, um a um. O ponto laranja marca o limiar padrão, 0,5, da matriz de confusão. A área sob a curva azul, 0,9496, está na legenda.
fpr_05 = fp_05 / n_neg

fig, ax = plt.subplots()
ax.plot(
    fpr,
    tpr,
    color="C0",
    linewidth=2.2,
    label=f"curva ROC (regressão logística), AUC = {auc:.4f}".replace(".", ","),
)
ax.plot(
    [0, 1],
    [0, 1],
    color="#7A8894",
    linestyle=":",
    linewidth=1.4,
    label="palpite sem informação",
)
ax.scatter([fpr_05], [revocacao_05], color="C1", zorder=5, s=40)
ax.annotate(
    f"limiar 0,5 (FPR {fpr_05:.3f}, revocação {revocacao_05:.3f})".replace(".", ","),
    xy=(fpr_05, revocacao_05),
    xytext=(0.05, 0.62),
    textcoords="data",
    fontsize=9,
    arrowprops=dict(arrowstyle="-", color="C1", linewidth=0.8),
)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xlabel("taxa de falso alarme (FPR)")
ax.set_ylabel("revocação (TPR)")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

O ponto marcado é o limiar 0,5 da matriz de confusão: FPR de 0,004 (os 40 falsos alarmes sobre 9.627 + 40 = 9.667 adimplentes) e revocação de 0,315 (os 105 acertos sobre 105 + 228 = 333 inadimplentes). É o mesmo par de números da matriz, agora como um ponto sobre a curva inteira. A diagonal pontilhada marca o que um classificador sem informação produziria: ganhar revocação só à custa de FPR, na mesma proporção. A curva do modelo logístico nunca passa para baixo dela, e é a distância entre as duas, resumida na área de 0,9496, que separa um classificador que aprendeu algo de um que está adivinhando.

A matriz de confusão, a precisão, a revocação e a curva ROC não dependem de qual classificador as produziu. A seção 9.4 deixou em aberto como LDA, QDA e Naive Bayes erram sobre o `Default`, e com esse vocabulário a pergunta tem resposta. No limiar 0,5, os quatro classificadores:

In [ ]:
classificadores = {
    "logística": modelo,
    "LDA": LinearDiscriminantAnalysis().fit(X, y),
    "QDA": QuadraticDiscriminantAnalysis().fit(X, y),
    "Naive Bayes": GaussianNB().fit(X, y),
}
tabela_quatro = pd.DataFrame(
    {
        nome: {
            "precisão": precision_score(y, m.predict(X)),
            "revocação": recall_score(y, m.predict(X)),
            "erro total": (m.predict(X) != y).mean(),
        }
        for nome, m in classificadores.items()
    }
).T.round(4)
tabela_quatro

In [ ]:
maior_revocacao = float(tabela_quatro["revocação"].max())
menor_revocacao = float(tabela_quatro["revocação"].min())
todos_perdem_a_maioria = bool(maior_revocacao < 0.5)
faixa_erro_total = (
    float(tabela_quatro["erro total"].min()),
    float(tabela_quatro["erro total"].max()),
)

menor_revocacao, maior_revocacao, todos_perdem_a_maioria, faixa_erro_total

> **🔧 Função**
>
> **`precision_score(y, previsto)`** e **`recall_score(y, previsto)`**: a precisão e a revocação da classe 1, calculadas direto das previsões, sem montar a matriz de confusão.

Os erros totais ficam entre 2,68% e 2,93%, mas os quatro dividem os erros de jeitos diferentes. A revocação vai de 0,2372 a 0,3153 entre eles, e `todos_perdem_a_maioria` confirma que nenhum acha metade dos inadimplentes no limiar 0,5. Para qualquer um dos quatro, baixar o limiar é o que muda esse quadro, e o preço é do mesmo tipo que a varredura mostrou para a logística: mais alarmes falsos.

## Comparando os Métodos

> **📌 Nota**
>
> Esta seção corresponde à seção 4.5 de James et al. (2023).

Cinco métodos de classificação apareceram até aqui: a regressão logística da seção 9.2, os três modelos generativos da 9.4 (LDA, QDA e Naive Bayes) e o *k*-NN, que as seções 7.3, 7.7 e 8.7 aplicaram à regressão e à classificação. Qual deles usar? O *k*-NN entra duas vezes, com *k* = 1 e com *k* = 15, tratados como dois classificadores distintos, e isso dá **seis classificadores**. A resposta honesta é "depende", e esta seção mostra de quê, com cenários simulados em que a fronteira verdadeira entre as duas classes é conhecida, porque quem simula escolhe a distribuição de cada classe. Em dois cenários essa fronteira é linear, e no terceiro, não.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from sklearn.discriminant_analysis import (
    LinearDiscriminantAnalysis,
    QuadraticDiscriminantAnalysis,
)
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

plt.style.use("estilo-figuras.mplstyle")

base = pd.read_csv("dados/Default.csv")
y_default = (base["inadimplente"] == "sim").astype(int)
X_default = base[["saldo", "renda"]].copy()
X_default["estudante_sim"] = (base["estudante"] == "sim").astype(int)

k_pequeno, k_grande = 1, 15

### Seis classificadores, três cenários

O *k* de cada *k*-NN é fixo e declarado de antemão (`k_pequeno = 1` e `k_grande = 15`), e não ajustado ao próprio dado. Cada valor de *k* é, na prática, um classificador diferente, e entra na tabela como tal.

Como na seção 9.4, um sorteio de treino só não compara métodos de forma confiável: o que decide é a média sobre muitos sorteios, contra um conjunto de teste grande e fixo. O teste é grande e fixo dentro de cada cenário para que a comparação não dependa de qual teste calhou; a variação que importa é a do sorteio de **treino**. São `n_teste_por_classe = 5000` pontos de teste por classe e `n_replicas = 200` sorteios de treino, o mesmo número de sorteios da seção 9.4.

In [ ]:
rng = np.random.default_rng(9)

media_a = np.array([-1.0, 0.0])
media_b = np.array([1.0, 0.0])

cov_independente = np.eye(2)
cov_correlacionada = np.array([[1.0, 0.8], [0.8, 1.0]])
cov_a_curva = np.array([[1.0, 0.7], [0.7, 1.0]])
cov_b_curva = np.array([[1.0, -0.7], [-0.7, 1.0]])

n_teste_por_classe = 5000
n_replicas = 200


def gera_duas_classes(cov_a, cov_b, n, rng):
    a = rng.multivariate_normal(media_a, cov_a, size=n)
    b = rng.multivariate_normal(media_b, cov_b, size=n)
    X = np.vstack([a, b])
    y = np.concatenate([np.zeros(n), np.ones(n)])
    return X, y


def constroi_metodos():
    return {
        "Logística": LogisticRegression(C=np.inf, max_iter=10000),
        "LDA": LinearDiscriminantAnalysis(),
        "QDA": QuadraticDiscriminantAnalysis(),
        "Naive Bayes": GaussianNB(),
        f"k-NN (k={k_pequeno})": KNeighborsClassifier(n_neighbors=k_pequeno),
        f"k-NN (k={k_grande})": KNeighborsClassifier(n_neighbors=k_grande),
    }


nomes_metodos = list(constroi_metodos().keys())

# fronteira curva usa mais dado de treino: k-NN só compensa a variância que
# sua flexibilidade custa quando há dado suficiente (seções 7.3 e 8.7).
cenarios = {
    "linear, preditores independentes": (cov_independente, cov_independente, 20),
    "linear, preditores correlacionados": (cov_correlacionada, cov_correlacionada, 20),
    "fronteira curva": (cov_a_curva, cov_b_curva, 200),
}

linhas = []
for nome_cenario, (cov_a, cov_b, n_treino_por_classe) in cenarios.items():
    X_teste, y_teste = gera_duas_classes(cov_a, cov_b, n_teste_por_classe, rng)
    for replica in range(n_replicas):
        X_treino, y_treino = gera_duas_classes(cov_a, cov_b, n_treino_por_classe, rng)
        for nome_metodo, modelo in constroi_metodos().items():
            modelo.fit(X_treino, y_treino)
            erro = 1 - modelo.score(X_teste, y_teste)
            treino_perfeito = modelo.score(X_treino, y_treino) == 1
            linhas.append((nome_cenario, nome_metodo, replica, erro, treino_perfeito))

resultados = pd.DataFrame(
    linhas, columns=["cenario", "metodo", "replica", "erro", "treino_perfeito"]
)
tabela_erro_medio = (
    resultados.groupby(["cenario", "metodo"])["erro"]
    .mean()
    .unstack("metodo")
    .reindex(index=list(cenarios.keys()), columns=nomes_metodos)
)
tabela_erro_medio.round(4)

> **🔧 Função**
>
> **`KNeighborsClassifier(n_neighbors)`**: o *k*-NN para classificação, que prevê a classe mais comum entre os `n_neighbors` vizinhos mais próximos.
>
> **`df.reindex(index, columns)`**: reordena as linhas e as colunas na ordem dada, aqui a dos cenários e a dos métodos.
>
> **`np.eye(2)`**: a matriz identidade 2 × 2, com 1 na diagonal e 0 fora dela. Como covariância, quer dizer variância 1 em cada preditor e nenhuma correlação entre eles.

Em cada cenário, a ordem dos seis pelo erro médio, do menor para o maior:

In [ ]:
{
    cenario: " < ".join(tabela_erro_medio.loc[cenario].sort_values().index)
    for cenario in cenarios
}

Os três cenários têm dois preditores e duas classes, com médias `media_a = (-1, 0)` e `media_b = (1, 0)`: só a primeira coordenada separa as classes, e a segunda tem a mesma distribuição nas duas. O que muda de um cenário a outro é a covariância dentro de cada classe. No primeiro, ela é a mesma nas duas classes, sem correlação. No segundo, é a mesma nas duas, com correlação 0,8. No terceiro, as correlações têm sinais opostos, 0,7 e -0,7, e só essa terceira mudança torna a fronteira de Bayes curva em vez de reta. (É a mesma diferença de covariância da seção 9.4, mas com as médias mais próximas, ±1,0 numa coordenada só contra ±1,5 nas duas, e por isso as classes se sobrepõem mais.) Em cada cenário, 200 sorteios de treino refazem o ajuste dos seis classificadores contra o mesmo teste.

In [ ]:
# Figura: Distribuição do erro de teste em 200 sorteios de treino, para os seis classificadores, nos três cenários simulados (um painel por cenário, cada um com sua própria escala de erro). Em laranja, o classificador de menor erro médio naquele cenário; em azul, os demais.
ordem_metodos = [
    "LDA",
    "Logística",
    "QDA",
    "Naive Bayes",
    f"k-NN (k={k_pequeno})",
    f"k-NN (k={k_grande})",
]
vencedor_por_cenario = {
    nome_cenario: tabela_erro_medio.loc[nome_cenario].idxmin()
    for nome_cenario in cenarios
}
titulo_por_cenario = {
    "linear, preditores independentes": "linear, independentes",
    "linear, preditores correlacionados": "linear, correlacionados",
    "fronteira curva": "fronteira curva",
}

fig, eixos = plt.subplots(3, 1, figsize=(6.4, 10))
for ax, nome_cenario in zip(eixos, cenarios.keys()):
    dados_cenario = [
        resultados.loc[
            (resultados["cenario"] == nome_cenario) & (resultados["metodo"] == m),
            "erro",
        ]
        for m in ordem_metodos
    ]
    caixas = ax.boxplot(
        dados_cenario,
        orientation="horizontal",
        tick_labels=ordem_metodos,
        widths=0.6,
        medianprops=dict(linewidth=1.8),
        boxprops=dict(linewidth=1.8),
        whiskerprops=dict(linewidth=1.4),
        capprops=dict(linewidth=1.4),
        flierprops=dict(markersize=3),
    )
    for i, nome_metodo in enumerate(ordem_metodos):
        cor = (
            "#D9480F"
            if nome_metodo == vencedor_por_cenario[nome_cenario]
            else "#4195D1"
        )
        caixas["boxes"][i].set_color(cor)
        caixas["medians"][i].set_color(cor)
        caixas["whiskers"][2 * i].set_color(cor)
        caixas["whiskers"][2 * i + 1].set_color(cor)
        caixas["caps"][2 * i].set_color(cor)
        caixas["caps"][2 * i + 1].set_color(cor)
        caixas["fliers"][i].set_markeredgecolor(cor)
    ax.set_title(titulo_por_cenario[nome_cenario])
    ax.set_xlabel("erro de teste")

legenda_cores = [
    Line2D([], [], color="#4195D1", linewidth=1.8, label="demais classificadores"),
    Line2D(
        [], [], color="#D9480F", linewidth=1.8, label="menor erro médio no cenário"
    ),
]
fig.legend(
    handles=legenda_cores,
    loc="lower center",
    ncols=2,
    bbox_to_anchor=(0.5, -0.03),
    frameon=False,
)
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> **`caixas["boxes"][i].set_color(cor)`**: o `ax.boxplot` da seção 9.1, aqui deitado (`orientation="horizontal"`) e com o nome de cada série no eixo (`tick_labels`), devolve cada parte desenhada (`boxes`, `medians`, `whiskers`...). `set_color` e `set_markeredgecolor` repintam uma parte depois de desenhada, e é assim que o laço destaca a caixa do vencedor.

A tabela dá a média de cada caixa; a figura dá a caixa inteira, que mostra quanto o erro de cada método varia de sorteio para sorteio. Quando a caixa de um método fica inteira de um lado da de outro, a vantagem raramente se inverte de um sorteio para outro. Quando as caixas se tocam, quem decide é a comparação sorteio a sorteio, feita mais adiante, porque todos os métodos usam o mesmo treino em cada sorteio:

In [ ]:
def quartis(cenario, metodo):
    erros = resultados.loc[
        (resultados["cenario"] == cenario) & (resultados["metodo"] == metodo), "erro"
    ]
    return erros.quantile(0.25), erros.quantile(0.75)


grupo_linear = ["LDA", "Logística", "QDA", "Naive Bayes"]
outros_curva = [m for m in nomes_metodos if m != "QDA"]

q_indep = {m: quartis("linear, preditores independentes", m) for m in grupo_linear}
grupo_se_sobrepoe_indep = max(q[0] for q in q_indep.values()) < min(
    q[1] for q in q_indep.values()
)

q_corr = {m: quartis("linear, preditores correlacionados", m) for m in grupo_linear}
nb_descolado_corr = q_corr["Naive Bayes"][0] > max(
    q_corr[m][1] for m in ["LDA", "Logística", "QDA"]
)

q3_qda_curva = quartis("fronteira curva", "QDA")[1]
qda_descolada_curva = q3_qda_curva < min(
    quartis("fronteira curva", m)[0] for m in outros_curva
)

bool(grupo_se_sobrepoe_indep), bool(nb_descolado_corr), bool(qda_descolada_curva)

No primeiro painel, as caixas de LDA, logística, QDA e Naive Bayes se sobrepõem: há uma faixa de erro comum às quatro (`grupo_se_sobrepoe_indep`). No segundo, a caixa do Naive Bayes se descola por completo, à direita das de LDA, logística e QDA (`nb_descolado_corr`). No terceiro, a caixa da QDA fica inteira à esquerda das outras cinco (`qda_descolada_curva`).

### Fronteira linear, preditores independentes

In [ ]:
piv_indep = resultados[
    resultados["cenario"] == "linear, preditores independentes"
].pivot(index="replica", columns="metodo", values="erro")

media_lda_indep = float(
    tabela_erro_medio.loc["linear, preditores independentes", "LDA"]
)
media_log_indep = float(
    tabela_erro_medio.loc["linear, preditores independentes", "Logística"]
)
media_nb_indep = float(
    tabela_erro_medio.loc["linear, preditores independentes", "Naive Bayes"]
)
media_qda_indep = float(
    tabela_erro_medio.loc["linear, preditores independentes", "QDA"]
)
media_knn1_indep = float(
    tabela_erro_medio.loc["linear, preditores independentes", f"k-NN (k={k_pequeno})"]
)
media_knnk_indep = float(
    tabela_erro_medio.loc["linear, preditores independentes", f"k-NN (k={k_grande})"]
)

margem_lda_qda_indep = media_qda_indep - media_lda_indep
fracao_lda_vence_qda_indep = float((piv_indep["LDA"] < piv_indep["QDA"]).mean())
lda_nb_mais_perto_que_lda_qda = bool(
    abs(media_nb_indep - media_lda_indep) < margem_lda_qda_indep
)
amplitude_grupo_lider_indep = max(
    media_lda_indep, media_log_indep, media_nb_indep, media_qda_indep
) - min(media_lda_indep, media_log_indep, media_nb_indep, media_qda_indep)

(
    round(media_lda_indep, 4),
    round(media_log_indep, 4),
    round(media_nb_indep, 4),
    round(media_qda_indep, 4),
    round(media_knnk_indep, 4),
    round(media_knn1_indep, 4),
    round(margem_lda_qda_indep, 4),
    round(fracao_lda_vence_qda_indep, 2),
    round(amplitude_grupo_lider_indep, 4),
    lda_nb_mais_perto_que_lda_qda,
    round(float((piv_indep["LDA"] < piv_indep["Naive Bayes"]).mean()), 2),
)

> **🔧 Função**
>
> **`df.pivot(index, columns, values)`**: reorganiza uma tabela longa numa larga, com uma linha por valor de `index` e uma coluna por valor de `columns`. Aqui, uma linha por sorteio e uma coluna por método, para comparar os métodos sorteio a sorteio.

Vinte pontos de treino por classe, preditores sem correlação dentro de cada classe e a mesma covariância nas duas: exatamente a suposição da LDA. LDA, Naive Bayes, logística e QDA, nessa ordem, ficam num grupo apertado, com 17,43%, 17,58%, 17,59% e 17,97% de erro médio; entre o primeiro e o último dos quatro, 0,0054. O *k*-NN com *k* = 15 vem logo depois, com 18,04%, e o *k*-NN com *k* = 1 fica em último, com 23,74%: 23,74 − 17,97 = 5,77 pontos percentuais acima da QDA, o preço de uma vizinhança de um só ponto quando o treino tem só vinte por classe.

Dentro desse grupo apertado, uma comparação se sustenta: a LDA erra menos que a QDA em 81% dos 200 sorteios, uma vantagem sistemática mas modesta, sobre uma margem estreita entre as médias (0,0054). Entre LDA e Naive Bayes, a distância é menor ainda (`lda_nb_mais_perto_que_lda_qda`), e a LDA erra menos em só 59% dos sorteios, pouco mais que cara ou coroa: pequena demais para separar os dois. Por que o Naive Bayes chega tão perto aqui? O cenário seguinte responde.

### Fronteira ainda linear, preditores correlacionados

In [ ]:
piv_corr = resultados[
    resultados["cenario"] == "linear, preditores correlacionados"
].pivot(index="replica", columns="metodo", values="erro")

media_lda_corr = float(
    tabela_erro_medio.loc["linear, preditores correlacionados", "LDA"]
)
media_qda_corr = float(
    tabela_erro_medio.loc["linear, preditores correlacionados", "QDA"]
)
media_log_corr = float(
    tabela_erro_medio.loc["linear, preditores correlacionados", "Logística"]
)
media_nb_corr = float(
    tabela_erro_medio.loc["linear, preditores correlacionados", "Naive Bayes"]
)

fracao_lda_vence_nb_corr = float((piv_corr["LDA"] < piv_corr["Naive Bayes"]).mean())

logistica_corr = resultados[
    (resultados["cenario"] == "linear, preditores correlacionados")
    & (resultados["metodo"] == "Logística")
]
fracao_treino_separavel_corr = float(logistica_corr["treino_perfeito"].mean())

diferenca_log_lda = (piv_corr["Logística"] - piv_corr["LDA"]).to_numpy()
separavel = logistica_corr.sort_values("replica")["treino_perfeito"].to_numpy()
diferenca_nos_separaveis = float(diferenca_log_lda[separavel].mean())
diferenca_nos_nao_separaveis = float(diferenca_log_lda[~separavel].mean())

(
    round(media_lda_corr, 4),
    round(media_qda_corr, 4),
    round(media_log_corr, 4),
    round(media_nb_corr, 4),
    round(fracao_lda_vence_nb_corr, 2),
    round(fracao_treino_separavel_corr, 2),
    round(diferenca_nos_separaveis, 4),
    round(diferenca_nos_nao_separaveis, 4),
)

A fronteira continua linear, com as mesmas médias; a única mudança é a correlação dentro de cada classe, 0,8 em vez de zero. LDA, QDA e logística caem para 5,15%, 5,49% e 6,06%. A correlação é a mesma nas duas classes, e sozinha não separa nada; o que ela faz é tornar $x_2$ útil. $x_2$ não distingue as classes, mas prevê parte do ruído de $x_1$ dentro de cada classe, e descontar esse ruído deixa as classes mais afastadas. LDA, QDA e logística aproveitam isso: a direção que melhor separa as classes, $\Sigma^{-1}(\mu_b - \mu_a)$, deixa de apontar só para a coordenada que muda de média e passa a misturar a outra também, mesmo ela tendo a mesma distribuição nas duas classes. O Naive Bayes não tem como enxergar isso, porque trata as duas coordenadas como independentes por definição.

A logística fica um pouco atrás da LDA. Uma causa possível é a separação: com 20 pontos por classe e classes tão bem separadas, em 45% dos sorteios o treino fica perfeitamente separável por uma reta, e a logística acerta todos os 40 pontos. Nesse caso a logística sem penalização não tem ótimo finito, e os coeficientes crescem até o otimizador parar. Mas separando os sorteios, a logística erra em média 0,0108 a mais que a LDA nos sorteios separáveis e 0,0077 a mais nos que não são: a desvantagem não vem só dos treinos perfeitamente separáveis. Aqui a suposição da LDA, normal com covariância comum, é exatamente a verdadeira, e com só 20 pontos por classe usar essa suposição rende mais do que estimar a fronteira sem supor nada sobre a distribuição de $X$. As duas razões que a seção 9.4 citou, as classes bem separadas e a normalidade com pouco dado, valem aqui ao mesmo tempo, e esta divisão dos sorteios não basta para separar o peso de cada uma.

No cenário anterior, o Naive Bayes errava 17,58%, dentro do grupo apertado. Aqui, com a suposição de independência agora violada, ele cai pouco, de 17,58% para 15,95%, enquanto LDA, QDA e logística caem para 5,15% a 6,06%. Quando a suposição de independência vale, ela quase não separa o Naive Bayes dos outros; quando não vale, são os outros que se afastam dele. A LDA erra menos que o Naive Bayes em 100% dos 200 sorteios.

### Fronteira curva

In [ ]:
piv_curva = resultados[resultados["cenario"] == "fronteira curva"].pivot(
    index="replica", columns="metodo", values="erro"
)

media_qda_curva = float(tabela_erro_medio.loc["fronteira curva", "QDA"])
media_lda_curva = float(tabela_erro_medio.loc["fronteira curva", "LDA"])
media_log_curva = float(tabela_erro_medio.loc["fronteira curva", "Logística"])
media_nb_curva = float(tabela_erro_medio.loc["fronteira curva", "Naive Bayes"])
media_knnk_curva = float(
    tabela_erro_medio.loc["fronteira curva", f"k-NN (k={k_grande})"]
)
media_knn1_curva = float(
    tabela_erro_medio.loc["fronteira curva", f"k-NN (k={k_pequeno})"]
)

metodos_sem_curva = ["LDA", "Logística", "Naive Bayes"]
melhor_linear_curva = tabela_erro_medio.loc[
    "fronteira curva", metodos_sem_curva
].idxmin()
media_melhor_linear_curva = float(
    tabela_erro_medio.loc["fronteira curva", melhor_linear_curva]
)

fracao_qda_vence_knnk_curva = float(
    (piv_curva["QDA"] < piv_curva[f"k-NN (k={k_grande})"]).mean()
)
fracao_knnk_vence_melhor_linear_curva = float(
    (piv_curva[f"k-NN (k={k_grande})"] < piv_curva[melhor_linear_curva]).mean()
)
margem_qda_proximo_curva = media_knnk_curva - media_qda_curva
margem_knnk_grupo_linear_curva = media_melhor_linear_curva - media_knnk_curva

(
    round(media_qda_curva, 4),
    round(media_knnk_curva, 4),
    round(media_log_curva, 4),
    round(media_lda_curva, 4),
    round(media_nb_curva, 4),
    round(media_knn1_curva, 4),
    round(fracao_qda_vence_knnk_curva, 2),
    round(fracao_knnk_vence_melhor_linear_curva, 2),
    round(margem_qda_proximo_curva, 4),
    round(margem_knnk_grupo_linear_curva, 4),
)

Este cenário usa 200 pontos de treino por classe, e não os 20 dos anteriores. Com covariâncias diferentes entre as classes, a fronteira de Bayes se curva, e tanto a QDA quanto o *k*-NN precisam de mais dado para que a flexibilidade compense a variância que ela custa; as seções 7.3 e 8.7 mostraram esse preço para regressão, e a seção 7.7, para classificação. Pela ordenação acima, a QDA tem o menor erro médio, 13,31%, e o *k*-NN com *k* = 15 vem em segundo, com 14,60%. Logística, LDA e Naive Bayes vêm logo atrás, os três com 15,65% no arredondamento a quatro casas, e o *k*-NN com *k* = 1 volta a ficar em último, com 17,23%.

As duas comparações que separam os grupos são sólidas. A QDA erra menos que o *k*-NN com *k* = 15, o segundo colocado, em 99% dos 200 sorteios, com uma margem de 0,0128 entre as médias. E o *k*-NN com *k* = 15 erra menos que o melhor de logística, LDA e Naive Bayes em 94% dos sorteios, com margem de 0,0105. Esses três não acompanham a curva deste cenário por motivos diferentes: logística e LDA só traçam retas, e a fronteira do Naive Bayes, que até pode se curvar quando as variâncias das classes diferem, não enxerga a correlação, e é só na correlação que as covariâncias das duas classes diferem aqui. Os dois que acompanham a curva sem pagar variância demais, a QDA pela forma quadrática certa e o *k*-NN com *k* = 15 por uma vizinhança grande o bastante, ficam à frente. O *k*-NN com *k* = 1 também se curva, mas com um vizinho só paga tanta variância que fica em último.

### Os seis classificadores sobre o `Default`

No `Default`, os seis classificadores usam as mesmas linhas e os mesmos preditores da seção 9.4: `saldo`, `renda` e o indicador de `estudante`. Antes de ajustar os *k*-NN, vale olhar a escala de cada preditor:

In [ ]:
X_default[["saldo", "renda"]].agg(["mean", "std"]).round(1)

`saldo` tem desvio padrão de 483,7 dólares, e `renda`, de 13.336,6, cerca de 27,6 vezes maior (13.336,6 / 483,7), mesmo as duas estando em dólares. O *k*-NN decide quem é "vizinho mais próximo" por distância, e nessa escala a distância seria dominada por `renda`, como a seção 7.3 viu com escolaridade e senioridade em `Income2`. Por isso os dois *k*-NN desta tabela entram num `Pipeline` com `StandardScaler`, para que `saldo` pese na conta.

In [ ]:
metodos_default = {
    "Logística": LogisticRegression(C=np.inf, solver="newton-cholesky", tol=1e-8),
    "LDA": LinearDiscriminantAnalysis(),
    "QDA": QuadraticDiscriminantAnalysis(),
    "Naive Bayes": GaussianNB(),
    f"k-NN (k={k_pequeno})": make_pipeline(
        StandardScaler(), KNeighborsClassifier(n_neighbors=k_pequeno)
    ),
    f"k-NN (k={k_grande})": make_pipeline(
        StandardScaler(), KNeighborsClassifier(n_neighbors=k_grande)
    ),
}

erros_default = {}
for nome_metodo, modelo in metodos_default.items():
    modelo.fit(X_default, y_default)
    erros_default[nome_metodo] = float((modelo.predict(X_default) != y_default).mean())

tabela_default = pd.DataFrame(
    {"erro sobre as próprias linhas do ajuste": erros_default}
).reindex(nomes_metodos)
tabela_default.round(4)

> **🔧 Função**
>
> **`make_pipeline(StandardScaler(), KNeighborsClassifier(...))`**: encadeia a padronização e o classificador num objeto só. `fit` padroniza cada preditor (média 0, desvio 1) com as médias e os desvios do próprio treino e ajusta o *k*-NN sobre o resultado; `predict` repete a mesma padronização antes de prever.
>
> **`df.agg(["mean", "std"])`**: aplica as funções listadas a cada coluna, uma linha por função.

Como nas seções 9.4 e 9.5, o erro aqui é sobre as mesmas linhas que ajustaram cada modelo, e não sobre dado novo. Os quatro que não são *k*-NN ficam lado a lado:

In [ ]:
quatro = tabela_default.squeeze()[["Logística", "LDA", "QDA", "Naive Bayes"]]
round(float(quatro.min()), 4), round(float(quatro.max()), 4)

Logística, LDA, QDA e Naive Bayes ficam entre 2,68% e 2,93%. O *k*-NN com *k* = 1 erra 0%: cada cliente é o seu próprio vizinho mais próximo nas 10.000 linhas do ajuste, e a previsão sempre repete o rótulo que ele já tinha. É o mesmo *k* = 1 que decorou o treino nas seções 7.3, 7.7 e 8.7, e ele não diz nada sobre um cliente novo.

In [ ]:
knnk_sem_escala = KNeighborsClassifier(n_neighbors=k_grande).fit(X_default, y_default)
erro_knnk_sem_escala = float((knnk_sem_escala.predict(X_default) != y_default).mean())
erro_knnk_com_escala = erros_default[f"k-NN (k={k_grande})"]

round(erro_knnk_sem_escala, 4), round(erro_knnk_com_escala, 4)

O *k*-NN com *k* = 15 mostra o preço da escala: sem `StandardScaler`, erra 3,18%; com ele, 2,59%, uma diferença de 3,18 − 2,59 = 0,59 ponto percentual só por pôr `saldo` e `renda` na mesma régua antes de medir distância.

In [ ]:
tabela_default.drop(index=f"k-NN (k={k_pequeno})").squeeze().idxmin()

> **🔧 Função**
>
> **`df.drop(index=rotulo)`**: devolve a tabela sem a linha `rotulo`. **`.squeeze()`** transforma uma tabela de uma coluna só numa `Series`, e aí `idxmin()` dá o rótulo do menor valor.

Fora o *k* = 1, o menor erro da tabela é o do *k*-NN com *k* = 15, 2,59%, abaixo dos quatro que ficam entre 2,68% e 2,93%. O mecanismo é o mesmo do *k* = 1, só que menos óbvio: com as mesmas 10.000 linhas usadas para ajustar e para prever, cada cliente está entre os seus próprios 15 vizinhos mais próximos, e isso puxa a previsão para o rótulo que ele já tem. Quanto isso pesa? Dá para medir tirando cada cliente da própria vizinhança:

In [ ]:
pipeline_knn15 = metodos_default[f"k-NN (k={k_grande})"]
X_padronizado = pipeline_knn15[0].transform(X_default)
vizinhos = pipeline_knn15[1].kneighbors(
    X_padronizado, n_neighbors=k_grande + 1, return_distance=False
)
rotulos_dos_vizinhos = y_default.to_numpy()[vizinhos[:, 1:]]
previsto_sem_o_proprio = (rotulos_dos_vizinhos.mean(axis=1) > 0.5).astype(int)
erro_sem_o_proprio = float((previsto_sem_o_proprio != y_default).mean())

round(erro_sem_o_proprio, 4)

> **🔧 Função**
>
> **`knn.kneighbors(X, n_neighbors, return_distance=False)`**: para cada linha de `X`, as posições dos `n_neighbors` pontos de treino mais próximos, do mais perto ao mais longe. A primeira coluna é o próprio ponto; `[:, 1:]` a descarta. **`pipeline[0]`** e **`pipeline[1]`** são os passos do `Pipeline`, aqui o `StandardScaler` e o *k*-NN.

Sem o próprio ponto, os 15 vizinhos mais próximos de cada cliente erram 2,78%, acima dos 2,68% da logística, e o *k*-NN deixa de ter o menor erro. A comparação pesa pouco contra o *k*-NN: a logística, com quatro parâmetros para 10.000 linhas, tem pouco a ganhar por prever as próprias linhas, enquanto o *k*-NN, que guarda cada linha, ganha muito. Nos cenários simulados, com um teste separado do treino, o *k*-NN com *k* = 15 não tem essa vantagem: perde para a LDA nos dois cenários lineares (18,04% contra 17,43%, e 10,98% contra 5,15%) e para a QDA na fronteira curva (14,60% contra 13,31%).

> **🔷 Conceito**
>
> | Cenário | O que se destacou | Do que depende |
> |---|---|---|
> | linear, preditores independentes | LDA, logística, QDA e Naive Bayes ficam num grupo apertado, sem vencedor destacado | nenhuma das quatro suposições é violada aqui |
> | linear, preditores correlacionados | LDA, QDA e logística caem juntos; Naive Bayes fica para trás | só o Naive Bayes assume preditores independentes dentro da classe |
> | fronteira curva, dado suficiente | QDA à frente; *k*-NN com *k* = 15 em segundo | acompanham a curva sem pagar variância demais |

Nenhum dos seis classificadores vence em todo cenário, e qual deles ajustar a um problema novo depende de uma fronteira que, fora de uma simulação, ninguém conhece. Para escolher entre eles sem essa informação, e para escolher o próprio *k* do *k*-NN em vez de fixá-lo, falta uma estimativa confiável do erro em dado novo, feita só com o dado disponível. É o que o capítulo 10 constrói.

## Leituras adicionais

- O [site oficial de James et al. (2023)](https://www.statlearning.com), com o PDF gratuito do livro, os dados usados neste capítulo e os laboratórios em Python.
- [`LogisticRegression`, na documentação do scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html), o estimador que ajusta a curva logística das seções 9.2 e 9.3, e que volta na comparação da seção 9.6.
- [`roc_curve`, na documentação do scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_curve.html), a função que a seção 9.5 usa para resumir todos os limiares de decisão possíveis numa curva só.

## Referências

- **James; Witten; Hastie; Tibshirani; Taylor**. *An Introduction to Statistical Learning with Applications in Python*. Springer. 2023.